# Polymarket — Closed Sports Markets: combined pipeline

One notebook that does the whole job end to end, using only Polymarket's official public APIs:

**discover closed Sports markets → collect hourly trade activity → one wide master CSV → one forecast per market**

On top of the original dataset it adds, for every market, **🆕 its event (`event_id` + event information)** and **🆕 a sport label for the event and for the market**.

## Legend — what was already there vs. what was added

| Marker | Meaning |
|---|---|
| ♻️ **UNCHANGED** | Copied verbatim from an original file (the file is named in the cell). The code is byte-identical. |
| ✏️ **MODIFIED** | Based on an original function. Every changed or added line carries a `# 🆕` comment. |
| 🆕 **NEW** | Did not exist in any of the original files. |

## 🆕 What was added (everything that was not in the original files)

1. **Event information for every market** — `event_id` plus the event's title, slug, ticker, description, dates, game fields (start time, score, …) and status. Captured from data the API already returns, stored in two new side files (`events.jsonl`, `market_events.jsonl`), with an automatic **backfill** for catalogues collected before this notebook existed. *(§3, §6)*
2. **Sport label for every event and market** — columns `event_sport`, `sport`, `sport_source`, and `sport_market_only` (the old label). Event-first method on top of the unchanged keyword rules, with a small `EVENT_OVERRIDE_RULES` list for known rule mistakes. *(§5)*
3. **New columns** — 36 in the master CSV (placed right after the original market columns) and 8 appended to the forecast CSV. *(§6, §7)*
4. **Events table** — `closed_sports_events.csv`, one row per event. *(§8)*
5. **One configuration cell and one "run" cell** instead of three scripts with command-line flags, plus an API sanity check that `tag_id = 1` is still "Sports". *(§1, §9)*
6. **Checks and audits** — file inventory, integrity checks, how each label was decided, an event-vs-old-label disagreement audit and a residual-unclassified audit. *(§10)*
7. **Safety details** — an old master CSV (no event columns) is detected and never overwritten without being asked; line breaks in event text are flattened in the CSVs so every row stays on one line; the master export keeps only an index of events in memory.

## Pipeline at a glance

| # | Stage | Writes (in `data/all_closed_sports/`) | Origin |
|---|---|---|---|
| 1 | **Discover** closed Sports markets, and save their events | `markets.jsonl`, 🆕 `events.jsonl`, 🆕 `market_events.jsonl` | ✏️ `collect_all_closed_sports.py` + 🆕 events |
| 2 | **Trades** → hourly aggregates for the last 24 h before close | `hourly_activity_24h/<market_id>.json.gz` | ♻️ `collect_all_closed_sports.py` |
| 3 | **Master CSV**, one row per market, 24 hourly bins as columns | `closed_sports_all_data.csv` | ✏️ `export_all_closed_sports_csv.py` + 🆕 event/sport columns |
| 4 | **Forecasts**, one row per usable settled binary market | `closed_sports_market_forecasts.csv` | ✏️ `prepare_sports_forecasts.py` + 🆕 event/sport columns |
| 5 | 🆕 **Events table**, one row per event | `closed_sports_events.csv` | 🆕 |

Sport labelling (§5) uses the keyword rules of `closed_sports_calibration.ipynb` (♻️ unchanged) with a 🆕 event-aware method on top.

## How to run

1. Open the notebook **from the project folder**. Data is written to `data/all_closed_sports/` — the same place the original scripts used, so anything they already collected (catalogue, hourly files, checkpoints) is picked up and **not downloaded again**.
2. **Smoke test first:** in the configuration cell set `MAX_PAGES = 3` (≈ 300 markets, a minute or so), run all cells, look at the results, then set it back to `None`. The full run simply continues from where the test stopped (set `FORCE_EXPORT = True` so the CSVs are rebuilt with the new data).
3. A full run takes hours (the trade stage dominates). Every stage is **resumable**: if you stop the cell or the kernel dies, run the same cell again.

> **Re-running on data collected with the old scripts:** the first `discover` call notices that markets have no event info and runs a one-off *event backfill* (§3). It only re-pages the market list — no trade requests.


## Design choices (and why)

1. **Same dataset as before.** The market universe, the API calls, the hourly aggregation, the resume/retry logic and the output file names are the original ones. Anything not marked 🆕 behaves exactly like the old scripts, and the original columns of every output keep their names, values and relative order. (This was verified by running the original scripts and this notebook on the same live slice and comparing the outputs.)
2. **Universe = closed markets with the official Sports tag** (`tag_id = 1`), paged with Gamma's keyset endpoint ordered by `id` — the definition the collection script already used, which keeps all previously collected data valid.
3. **Event info comes for free.** Every market returned by the Gamma API already embeds its `events` list. We save it instead of discarding it — **no extra API calls**. (The separate `/events` endpoint would embed every market of every event again — about 60 KB per event — and is far heavier.)
4. **Events live in side files, not inside `markets.jsonl`.** `markets.jsonl` stays byte-compatible with the old scripts, so an existing catalogue and the trade stage's row-index checkpoint remain valid. Events are stored once each (`events.jsonl`, many markets share one event) and linked by `market_events.jsonl`. Old catalogues are upgraded automatically by the backfill pass.
5. **Nothing about an event is lost, but the CSV stays readable.** `events.jsonl` holds the full embedded event (minus 3 image-URL fields). The CSVs get a curated flat subset (`event_title`, `event_slug`, `event_score`, …) listed in `EVENT_CSV_FIELDS`; edit that tuple to add or remove columns.
6. **New columns are inserted right after the original market columns** in the master CSV, so the original columns keep their order and the 288 hourly columns still come last. In the forecast CSV they are appended after the 12 original columns.
7. **Sport label: event first, original method as fallback** (§5). The event is the natural unit of a sport ("EPL: West Ham vs. Manchester City"), and its slug/title are far more reliable than keywords in a single market question. The old market-only label is kept as `sport_market_only`, so earlier results stay reproducible and the improvement is auditable.
8. **Labels are computed while writing the CSV**, from `events.jsonl` + the catalogue, so they never go stale relative to the data (and re-labelling after a rule change is just a re-export — no API calls).
9. **Large-data friendliness.** The export keeps only an `{event_id: byte offset}` index in memory and reads events on demand, instead of loading every event description into RAM.
10. **Derived files are not silently overwritten.** Like the original `--force` flags: an existing master/events CSV is kept unless `FORCE_EXPORT = True`; the forecast CSV is rebuilt automatically whenever the master CSV is newer.


---
# 1. Setup and configuration

🆕 **Configuration cell.** The three original scripts were driven by command-line flags (`--stage`, `--max-pages`, `--workers`, `--force`, …). In a notebook those become plain variables in one cell; the defaults equal the scripts' defaults. All other cells should not need editing.


In [ ]:
from __future__ import annotations

import csv
import gzip
import json
import os
import re
import sys
import tempfile
import time
from collections import Counter
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from datetime import datetime, timedelta, timezone
from functools import lru_cache
from pathlib import Path
from typing import Any
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import display

# ---------------------------------------------------------------------------------------------
# ♻️ UNCHANGED constants from collect_all_closed_sports.py.
# NOTE: `DATA` is the *Data API URL* (as in the original script) - the data *folder* is DATA_DIR.
# The original names are kept so the copied functions below work without any edits.
# ---------------------------------------------------------------------------------------------
GAMMA = 'https://gamma-api.polymarket.com'
DATA = 'https://data-api.polymarket.com'
SPORTS_TAG_ID = '1'
USER_AGENT = 'school-polymarket-sports-collector/1.0'

# ---------------------------------------------------------------------------------------------
# 🆕 CONFIGURATION - replaces the command-line flags of the original scripts.
# The defaults reproduce the original "full run". Change only this cell to control the pipeline.
# ---------------------------------------------------------------------------------------------
ROOT = Path.cwd()                                   # run the notebook from the project folder
DATA_DIR = ROOT / 'data' / 'all_closed_sports'      # ♻️ same folder as the original scripts, so any
                                                    #    data/checkpoints they already produced are reused

# Which stages to run (original: --stage discover | trades | all; export and prepare were separate scripts)
RUN_DISCOVERY    = True    # 1. catalogue of closed Sports markets (+ their events)
RUN_TRADES       = True    # 2. hourly trade aggregates per market  (the slow stage - hours/days for a full run)
RUN_EXPORT       = True    # 3. wide master CSV (one row per market)
RUN_FORECASTS    = True    # 4. one-forecast-per-market CSV
RUN_EVENTS_TABLE = True    # 5. 🆕 one-row-per-event CSV

# Discovery  (original: --page-size, --max-pages)
PAGE_SIZE = 100            # Gamma maximum
MAX_PAGES = None           # None = whole catalogue.  For a quick smoke test use e.g. 3  (= 300 markets)

# Trades  (original: --max-markets, --workers, --trade-limit, --keep-raw, --dry-run)
MAX_MARKETS = None         # None = every catalogued market.  Smoke test: e.g. 300
WORKERS = 4
TRADE_LIMIT = 10_000       # the Data API returns at most this many trades per market
KEEP_RAW = False           # True also archives every raw trade response (uses much more disk)
DRY_RUN = False            # True = only print what would be fetched; skips export/forecast stages

# Export / forecasts  (original: --force, --progress-every, --chunk-size)
FORCE_EXPORT = True       # rebuild the master CSV even if it exists (needed after collecting more trades)
FORCE_FORECASTS = False    # rebuild forecasts even if newer than the master CSV
PROGRESS_EVERY = 100_000
FORECAST_CHUNK_SIZE = 25_000

MASTER_CSV = DATA_DIR / 'closed_sports_all_data.csv'              # ♻️ same name as before
FORECASTS_CSV = DATA_DIR / 'closed_sports_market_forecasts.csv'   # ♻️ same name as before
EVENTS_CSV = DATA_DIR / 'closed_sports_events.csv'                # 🆕

# ♻️ the same argument checks the original collector did through argparse
if not 1 <= PAGE_SIZE <= 100:
    raise ValueError('PAGE_SIZE must be 1–100 (Gamma endpoint maximum)')
if not 1 <= WORKERS <= 12:
    raise ValueError('WORKERS must be 1–12')

print(f'Data folder: {DATA_DIR}')


Data folder: /Users/pietrorebecchi/Desktop/Final/data/all_closed_sports


---
# 2. Shared helpers  (♻️ UNCHANGED — `collect_all_closed_sports.py`)

* `get_json` — HTTP GET with bounded exponential back-off (public endpoints rate-limit).
* `jsonl_ids` — the set of `id`s already written to a `.jsonl` file; this is how re-runs skip finished work.
* `compact_market` — keeps only the research fields of a market and **drops** descriptions/media/the embedded `events` payload. It is unchanged, so `markets.jsonl` rows are identical to before; the events it drops are saved separately by the 🆕 code in §3.
* Atomic checkpoint readers/writers (write to `.tmp`, then rename) so a crash never leaves a half-written checkpoint.


In [2]:
# ♻️ UNCHANGED - copied from collect_all_closed_sports.py
def get_json(url: str, retries: int = 4):
    """Fetch JSON with bounded exponential backoff for public API rate limits."""
    error = None
    for attempt in range(retries):
        try:
            request = Request(url, headers={'User-Agent': USER_AGENT})
            with urlopen(request, timeout=45) as response:
                return json.load(response)
        except Exception as exc:  # HTTP errors also belong in the resumable log.
            error = exc
            if attempt + 1 < retries:
                time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f'{type(error).__name__}: {error}')


def jsonl_ids(path: Path) -> set[str]:
    if not path.exists():
        return set()
    ids = set()
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                try:
                    ids.add(str(json.loads(line)['id']))
                except (KeyError, json.JSONDecodeError):
                    continue
    return ids


MARKET_FIELDS = (
    'id', 'conditionId', 'slug', 'question', 'startDate', 'endDate', 'closedTime',
    'outcomes', 'outcomePrices', 'clobTokenIds', 'volume', 'volumeNum',
    'volumeClob', 'volumeAmm', 'liquidity', 'liquidityNum', 'liquidityClob',
    'liquidityAmm', 'spread', 'bestBid', 'bestAsk', 'lastTradePrice',
)


def compact_market(market: dict) -> dict:
    """Keep research variables, not large descriptions/media/event payloads."""
    return {field: market.get(field) for field in MARKET_FIELDS if field in market}


def read_discovery_checkpoint(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        return {}


def write_discovery_checkpoint(path: Path, checkpoint: dict) -> None:
    """Atomically persist the cursor only after its page is safely catalogued."""
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(checkpoint, separators=(',', ':')), encoding='utf-8')
    temporary.replace(path)


def read_trade_checkpoint(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        return {}


def write_trade_checkpoint(path: Path, checkpoint: dict) -> None:
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(checkpoint, separators=(',', ':')), encoding='utf-8')
    temporary.replace(path)

### Sanity check — is `tag_id = 1` really "Sports"?  (🆕)

The whole dataset is defined by this tag id, so verify it before downloading anything. This is a one-request check inspired by the first notebook (`Polymarket_Closed_Sports_Tag_Dataset.ipynb`), rewritten with `get_json` so the notebook does not need the `requests` package.


In [3]:
# 🆕 (idea from Polymarket_Closed_Sports_Tag_Dataset.ipynb, rewritten with get_json instead of `requests`)
# Fails loudly if tag_id 1 ever stops being "Sports" - otherwise the whole dataset would silently be wrong.
sports_tag = get_json(f'{GAMMA}/tags/{SPORTS_TAG_ID}')
print('Tag ID:', sports_tag.get('id'), '| Label:', sports_tag.get('label'), '| Slug:', sports_tag.get('slug'))
if str(sports_tag.get('id')) != SPORTS_TAG_ID or sports_tag.get('slug') != 'sports':
    raise RuntimeError(f'SPORTS_TAG_ID={SPORTS_TAG_ID} no longer refers to the Sports tag: {sports_tag}')

Tag ID: 1 | Label: Sports | Slug: sports


---
# 3. Stage 1 — Discover closed Sports markets and their events  (✏️ MODIFIED)

`discover()` is the original keyset-pagination loop. What is **🆕 new** in it:

| File | Content | Written |
|---|---|---|
| `events.jsonl` | one line per *unique* event: the full embedded event object (minus image URLs). Key: `id` | the first time an event is seen (first version wins) |
| `market_events.jsonl` | one line per market: `{"id": <market id>, "event_ids": [<event ids>]}` (the first id is the primary event) | for every market, even if it has no event (`[]`) |

**Event backfill.** If `markets.jsonl` has markets that are missing from `market_events.jsonl` (a catalogue made with the old script), the function re-pages the market list from the start *only to save the events*. It has its own checkpoint (`events_backfill_checkpoint.json`), never rewrites `markets.jsonl` (rows are only appended) and leaves the original discovery checkpoint alone.

**Crash safety.** Within a page the write order is *events → links → catalogue → checkpoint*, so after a crash the event files can be ahead of the catalogue but never behind it.


In [4]:
# 🆕 Which parts of the embedded event object we keep.
#
# events.jsonl stores the WHOLE embedded event, minus three image-URL fields (pure media, same reason the
# original compact_market() dropped market images). Nothing else is thrown away, so any event field can be
# added to the CSVs later without re-downloading anything.
EVENT_DROP_FIELDS = ('image', 'icon', 'featuredImage')

# The CSVs get a flat, human-friendly subset of those fields, prefixed with `event_`
# (e.g. event_title, event_slug, event_score). Delete 'description' from this tuple to shrink the master CSV
# (it is the longest text field and is repeated on every market of the event). Line breaks inside text fields are
# replaced by spaces in the CSVs (events.jsonl keeps the original text).
EVENT_CSV_FIELDS = (
    # identity / text
    'ticker', 'slug', 'title', 'description', 'resolutionSource', 'seriesSlug',
    # event timing as published by Polymarket
    'startDate', 'endDate', 'creationDate', 'closedTime',
    # sports-game fields that Polymarket fills for games (kick-off time, final score, ...)
    'startTime', 'eventDate', 'eventWeek', 'period', 'live', 'ended', 'score', 'elapsed', 'finishedTimestamp',
    # status flags and size snapshots (final values for closed events)
    'active', 'closed', 'archived', 'restricted', 'featured',
    'volume', 'liquidity', 'openInterest', 'negRisk', 'negRiskMarketID',
)

# Column groups that are added to the master CSV (placed right after the original market columns).
EVENT_COLUMNS = ['event_id', 'event_ids', 'event_count'] + [f'event_{field}' for field in EVENT_CSV_FIELDS]
SPORT_COLUMNS = ['event_sport', 'sport', 'sport_source', 'sport_market_only']


def compact_event(event: dict) -> dict:
    """🆕 Keep the full embedded event except media URLs; `id` is forced to str (it is the join key)."""
    kept = {field: value for field, value in event.items() if field not in EVENT_DROP_FIELDS}
    kept['id'] = str(event['id'])
    return kept


In [5]:
def discover(out: Path, page_size: int, max_pages: int | None, dry_run: bool):
    """Keyset-page through the public Sports catalogue and append unique rows.

    ✏️ MODIFIED from collect_all_closed_sports.discover(). Everything not marked 🆕 is the original logic.
    """
    catalogue = out / 'markets.jsonl'
    events_catalogue = out / 'events.jsonl'              # 🆕 one line per unique event
    market_events = out / 'market_events.jsonl'          # 🆕 one line per market: {"id": market, "event_ids": [...]}
    checkpoint_path = out / 'discovery_checkpoint.json'
    known = jsonl_ids(catalogue)
    known_events = jsonl_ids(events_catalogue)           # 🆕 (jsonl_ids reads the 'id' key; events have one)
    linked = jsonl_ids(market_events)                    # 🆕 markets whose events were already saved
    checkpoint = read_discovery_checkpoint(checkpoint_path)
    page = int(checkpoint.get('pages_completed', 0))
    cursor = checkpoint.get('next_cursor')
    seen_cursors = set()
    if dry_run:
        print(f'DRY RUN: would page closed markets with tag_id={SPORTS_TAG_ID} into {catalogue}')
        return
    # 🆕 BACKFILL. A catalogue collected with the old script has markets but no event files. In that case
    # page through the API once more from the beginning (cheap: ~100 markets per request, no trade calls) and
    # save the events. It has its own checkpoint, so an interruption resumes instead of starting over, and the
    # original discovery checkpoint is left untouched. markets.jsonl is only ever appended to (so the trade
    # stage's row-index checkpoint stays valid).
    backfill_needed = known - linked
    if backfill_needed:
        checkpoint_path = out / 'events_backfill_checkpoint.json'
        backfill = read_discovery_checkpoint(checkpoint_path)
        page = int(backfill.get('pages_completed', 0))
        cursor = backfill.get('next_cursor')
        print(f'{len(backfill_needed):,} catalogued markets have no event info yet -> event backfill pass '
              f'(starting at keyset page {page + 1})', flush=True)
    elif cursor:
        print(f'resuming discovery at keyset page {page + 1}', flush=True)
    while max_pages is None or page < max_pages:
        params = {'closed': 'true', 'tag_id': SPORTS_TAG_ID, 'limit': page_size,
                  'order': 'id', 'ascending': 'true'}
        if cursor:
            params['after_cursor'] = cursor
        response = get_json(f'{GAMMA}/markets/keyset?{urlencode(params)}')
        if not isinstance(response, dict):
            raise RuntimeError('Unexpected keyset response; expected an object with markets and next_cursor.')
        markets = response.get('markets', [])
        if not isinstance(markets, list):
            raise RuntimeError('Unexpected keyset response; markets was not a list.')
        if not markets:
            break
        added = 0
        # Write order matters for crash safety: events -> links -> catalogue -> (then) checkpoint.
        # A crash can therefore leave the event files *ahead* of the catalogue, never behind it.
        with events_catalogue.open('a', encoding='utf-8') as event_handle, \
                market_events.open('a', encoding='utf-8') as link_handle, \
                catalogue.open('a', encoding='utf-8') as handle:
            for market in markets:
                market_id = str(market.get('id', ''))
                if not market_id:
                    continue
                if market_id not in linked:                                           # 🆕
                    event_ids = []
                    for event in market.get('events') or []:
                        if not isinstance(event, dict) or event.get('id') in (None, ''):
                            continue
                        event_id = str(event['id'])
                        event_ids.append(event_id)
                        if event_id not in known_events:                              # events repeat across markets
                            event_handle.write(json.dumps(compact_event(event), separators=(',', ':')) + '\n')
                            known_events.add(event_id)
                    link_handle.write(json.dumps({'id': market_id, 'event_ids': event_ids},
                                                 separators=(',', ':')) + '\n')
                    linked.add(market_id)
                if market_id in known:
                    continue
                handle.write(json.dumps(compact_market(market), separators=(',', ':')) + '\n')
                known.add(market_id)
                added += 1
        page += 1
        print(f'discovery keyset_page={page} received={len(markets)} new={added} total={len(known)} '
              f'events={len(known_events)}', flush=True)
        next_cursor = response.get('next_cursor')
        write_discovery_checkpoint(checkpoint_path, {
            'pages_completed': page,
            'next_cursor': next_cursor,
            'markets_catalogued': len(known),
            'updated_at_unix': time.time(),
        })
        if not next_cursor:
            break
        if next_cursor in seen_cursors:
            raise RuntimeError('Keyset cursor repeated; refusing to loop over the same page.')
        seen_cursors.add(next_cursor)
        cursor = next_cursor
        time.sleep(.15)
    print(f'discovery complete: {len(known)} unique closed Sports markets in {catalogue}; '
          f'{len(known_events)} events in {events_catalogue}', flush=True)


---
# 4. Stage 2 — Hourly trade aggregates  (♻️ UNCHANGED — `collect_all_closed_sports.py`)

For each catalogued market the Data API trades are downloaded and **immediately reduced** to hourly bins for the last 24 hours before `endDate` (raw trades are discarded unless `KEEP_RAW = True`). Per hour and outcome token we keep: trade records, unique transactions / wallets, buy / sell counts, share size, notional (USDC), VWAP and median price.

Behaviour worth knowing (all original):
* **Resumable.** A row-index checkpoint (`trade_checkpoint.json`), a per-market result file and a retry journal (`trade_retry_events.jsonl`) mean a re-run only does the missing work; failed markets are retried once at the end of each run.
* **Truncation flag.** The API returns at most `TRADE_LIMIT` trades per market; a market that hits the limit is counted as `possibly_truncated` (it is not evidence the market had exactly that many trades).
* YES and NO tokens are kept in separate bins — mixing their prices would make an hourly probability meaningless.
* Liquidity and bid/ask spread are *not* estimated here (the trade endpoint has executions, not order-book snapshots).


In [6]:
# ♻️ UNCHANGED - copied from collect_all_closed_sports.py  (resume / retry bookkeeping)
def pending_retry_markets(retry_events_path: Path) -> dict[str, dict]:
    """Replay the retry journal to recover markets still needing a fetch."""
    pending: dict[str, dict] = {}
    if not retry_events_path.exists():
        return pending
    with retry_events_path.open(encoding='utf-8') as handle:
        for line in handle:
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                continue
            market_id = str(event.get('id', ''))
            if not market_id:
                continue
            if event.get('event') == 'error' and isinstance(event.get('market_ref'), dict):
                pending[market_id] = event['market_ref']
            elif event.get('event') == 'resolved':
                pending.pop(market_id, None)
    return pending


def legacy_log_frontier(log_path: Path, catalogue: Path) -> int:
    """Locate the furthest catalogue row touched by the pre-checkpoint run."""
    if not log_path.exists():
        return 0
    highest_id = -1
    with log_path.open(encoding='utf-8') as handle:
        for line in handle:
            try:
                market_id = int(json.loads(line).get('id', -1))
            except (json.JSONDecodeError, TypeError, ValueError):
                continue
            highest_id = max(highest_id, market_id)
    if highest_id < 0:
        return 0
    row_index = 0
    with catalogue.open(encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                row_index += 1
                continue
            try:
                market_id = int(json.loads(line).get('id', -1))
            except (json.JSONDecodeError, TypeError, ValueError):
                return row_index
            row_index += 1
            if market_id >= highest_id:
                return row_index
    return row_index


def bootstrap_trade_resume(out: Path, total_markets: int) -> tuple[dict, int]:
    """Migrate a legacy run by indexing missing outputs and its log frontier."""
    catalogue = out / 'markets.jsonl'
    hourly_dir = out / 'hourly_activity_24h'
    checkpoint_path = out / 'trade_checkpoint.json'
    retry_events_path = out / 'trade_retry_events.jsonl'
    checkpoint = read_trade_checkpoint(checkpoint_path)
    if checkpoint.get('resume_version') == 2:
        return checkpoint, min(int(checkpoint.get('next_catalogue_index', 0)), total_markets)

    frontier = min(legacy_log_frontier(out / 'trade_collection_log.jsonl', catalogue), total_markets)
    retry_events_path.parent.mkdir(parents=True, exist_ok=True)
    queued = 0
    # Old logs did not retain market metadata on failures. Inspect the already
    # attempted prefix once and queue only markets whose summary file is absent.
    with catalogue.open(encoding='utf-8') as source, retry_events_path.open('a', encoding='utf-8') as retry_log:
        for row_index, line in enumerate(source):
            if row_index >= frontier:
                break
            if not line.strip():
                continue
            try:
                market = json.loads(line)
            except json.JSONDecodeError:
                continue
            market_id = str(market.get('id', ''))
            if market_id and market.get('conditionId') and not (hourly_dir / f'{market_id}.json.gz').exists():
                retry_log.write(json.dumps({'event': 'error', 'id': market_id,
                                            'market_ref': market,
                                            'detail': 'queued during legacy resume migration'},
                                           separators=(',', ':')) + '\n')
                queued += 1
        retry_log.flush()
    checkpoint = {'resume_version': 2, 'next_catalogue_index': frontier,
                  'catalogue_size': total_markets, 'updated_at_unix': time.time()}
    write_trade_checkpoint(checkpoint_path, checkpoint)
    print(f'legacy run indexed: resume at catalogue row {frontier:,}/{total_markets:,}; '
          f'queued {queued:,} unsaved earlier markets for retry', flush=True)
    return checkpoint, frontier

In [7]:
# ♻️ UNCHANGED - copied from collect_all_closed_sports.py  (trades -> hourly bins)
def parse_utc(value: str | None) -> datetime | None:
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace('Z', '+00:00')).astimezone(timezone.utc)
    except ValueError:
        return None


def hourly_activity(market: dict, trades: list[dict], hours: int = 24) -> list[dict]:
    """Aggregate timestamped trades into the final `hours` pre-close bins.

    The returned trade endpoint contains executions, not historical order-book
    snapshots. Thus price and activity are valid retrospective hourly measures;
    liquidity and bid/ask spread are intentionally not estimated here.
    """
    close_time = parse_utc(market.get('endDate'))
    if close_time is None:
        return []
    close_unix = close_time.timestamp()
    # A binary market has separate YES and NO token trades.  Keep them apart:
    # combining their prices would make an hourly probability meaningless.
    buckets: dict[tuple[int, str, str], list[dict]] = {}
    for trade in trades:
        try:
            seconds_before = close_unix - float(trade['timestamp'])
            hour_before = int((seconds_before - 1e-9) // 3600) + 1
        except (KeyError, TypeError, ValueError):
            continue
        if 1 <= hour_before <= hours:
            key = (hour_before, str(trade.get('asset', '')), str(trade.get('outcome', '')))
            buckets.setdefault(key, []).append(trade)

    rows = []
    for (hour_before, asset, outcome), values in sorted(buckets.items()):
        sizes, notionals, prices = [], [], []
        wallets, transactions = set(), set()
        buy_records = sell_records = 0
        for trade in values:
            try:
                size, price = float(trade['size']), float(trade['price'])
            except (KeyError, TypeError, ValueError):
                continue
            sizes.append(size); prices.append(price); notionals.append(size * price)
            if trade.get('proxyWallet'):
                wallets.add(trade['proxyWallet'])
            if trade.get('transactionHash'):
                transactions.add(trade['transactionHash'])
            buy_records += trade.get('side') == 'BUY'
            sell_records += trade.get('side') == 'SELL'
        if not sizes:
            continue
        prices.sort()
        middle = len(prices) // 2
        median_price = prices[middle] if len(prices) % 2 else (prices[middle - 1] + prices[middle]) / 2
        rows.append({
            'market_id': str(market['id']),
            'condition_id': market.get('conditionId'),
            'slug': market.get('slug'),
            'question': market.get('question'),
            'end_time_utc': close_time.isoformat(),
            'hours_before_close': hour_before,
            'hour_start_utc': datetime.fromtimestamp(close_unix - hour_before * 3600, tz=timezone.utc).isoformat(),
            'hour_end_utc': datetime.fromtimestamp(close_unix - (hour_before - 1) * 3600, tz=timezone.utc).isoformat(),
            'asset': asset,
            'outcome': outcome,
            'outcome_index': values[0].get('outcomeIndex'),
            'trade_records': len(values),
            'unique_transaction_hashes': len(transactions),
            'unique_wallets_observed': len(wallets),
            'buy_records': buy_records,
            'sell_records': sell_records,
            'share_size': sum(sizes),
            # The endpoint can return both sides of an execution. This is a
            # trade-side total, not an exchange-verified market volume.
            'trade_side_notional_usdc': sum(notionals),
            'vwap_price': sum(notionals) / sum(sizes),
            'median_price': median_price,
        })
    return rows

In [8]:
# ♻️ UNCHANGED - copied from collect_all_closed_sports.py  (one market: cache check, fetch, aggregate)
def fetch_trade_file(market: dict, trades_dir: Path | None, hourly_dir: Path,
                     trade_limit: int, keep_raw: bool, dry_run: bool) -> dict:
    market_id = str(market['id'])
    activity_path = hourly_dir / f'{market_id}.json.gz'
    if activity_path.exists():
        try:
            with gzip.open(activity_path, 'rt', encoding='utf-8') as handle:
                json.load(handle)
            return {'id': market_id, 'status': 'cached'}
        except (OSError, EOFError, json.JSONDecodeError):
            # A hard kill during an older non-atomic write may leave a partial
            # file. Keep it for diagnosis, then replace it after a good fetch.
            corrupt_path = activity_path.with_suffix(activity_path.suffix + '.corrupt')
            if corrupt_path.exists():
                corrupt_path = activity_path.with_name(f'{market_id}.{int(time.time())}.json.gz.corrupt')
            activity_path.rename(corrupt_path)
    path = trades_dir / f'{market_id}.json.gz' if trades_dir else None
    if path and path.exists():
        # Raw files collected before hourly aggregation was added can be
        # upgraded locally without another API request.
        if not activity_path.exists() and not dry_run:
            try:
                with gzip.open(path, 'rt', encoding='utf-8') as handle:
                    payload = json.load(handle)
                with gzip.open(activity_path, 'wt', encoding='utf-8') as handle:
                    json.dump(hourly_activity(market, payload.get('trades', [])), handle, separators=(',', ':'))
                return {'id': market_id, 'status': 'cached', 'hourly_rebuilt': True}
            except Exception as exc:
                return {'id': market_id, 'status': 'error', 'detail': f'cached-file rebuild: {type(exc).__name__}: {exc}'}
        return {'id': market_id, 'status': 'cached'}
    condition_id = market.get('conditionId')
    if not condition_id:
        return {'id': market_id, 'status': 'skipped', 'detail': 'missing conditionId'}
    if dry_run:
        return {'id': market_id, 'status': 'would fetch'}
    try:
        query = urlencode({'market': condition_id, 'limit': trade_limit, 'takerOnly': 'false'})
        trades = get_json(f'{DATA}/trades?{query}')
        if not isinstance(trades, list):
            raise ValueError('trade response was not a JSON list')
        temporary_activity = activity_path.with_suffix(activity_path.suffix + '.tmp')
        with gzip.open(temporary_activity, 'wt', encoding='utf-8') as handle:
            json.dump(hourly_activity(market, trades), handle, separators=(',', ':'))
        temporary_activity.replace(activity_path)
        if keep_raw:
            payload = {'market_id': market_id, 'condition_id': condition_id,
                       'retrieved_at_unix': time.time(), 'trade_limit': trade_limit,
                       'history_possibly_truncated': len(trades) >= trade_limit, 'trades': trades}
            assert path is not None
            with gzip.open(path, 'wt', encoding='utf-8') as handle:
                json.dump(payload, handle, separators=(',', ':'))
        return {'id': market_id, 'status': 'downloaded', 'trades': len(trades),
                'possibly_truncated': len(trades) >= trade_limit}
    except Exception as exc:
        return {'id': market_id, 'status': 'error', 'detail': f'{type(exc).__name__}: {exc}'}

In [9]:
# ♻️ UNCHANGED - copied from collect_all_closed_sports.py  (thread pool, checkpoint, retries)
def collect_trades(out: Path, workers: int, trade_limit: int, max_markets: int | None,
                   keep_raw: bool, dry_run: bool):
    catalogue = out / 'markets.jsonl'
    if not catalogue.exists():
        raise FileNotFoundError(f'Missing {catalogue}; run --stage discover first.')
    with catalogue.open(encoding='utf-8') as handle:
        total_markets = sum(1 for line in handle if line.strip())
    if max_markets is not None:
        total_markets = min(total_markets, max_markets)
    trades_dir = out / 'trades' if keep_raw else None
    if trades_dir:
        trades_dir.mkdir(parents=True, exist_ok=True)
    hourly_dir = out / 'hourly_activity_24h'
    hourly_dir.mkdir(parents=True, exist_ok=True)
    log_path = out / 'trade_collection_log.jsonl'
    checkpoint_path = out / 'trade_checkpoint.json'
    retry_events_path = out / 'trade_retry_events.jsonl'
    checkpoint, start_index = bootstrap_trade_resume(out, total_markets)
    if max_markets is not None and start_index:
        raise ValueError('Cannot combine --max-markets with an existing trade checkpoint.')
    print(f'resuming at catalogue row {start_index + 1:,}/{total_markets:,} '
          f'(next row checkpointed: {start_index:,})', flush=True)

    summary = {'cached': 0, 'downloaded': 0, 'error': 0, 'skipped': 0,
               'would fetch': 0, 'possibly_truncated': 0}
    max_in_flight = max(workers * 4, workers)
    processed = 0
    next_index = start_index
    catalogue_index = 0  # Nonblank market rows; matches total_markets/checkpoint.
    pending_info: dict = {}
    completed_indexes: set[int] = set()
    retry_markets = pending_retry_markets(retry_events_path)

    def record_result(future, log, retry_log):
        nonlocal processed
        record = future.result()
        market, index, _was_retry = pending_info.pop(future)
        market_id = str(record.get('id', market.get('id', '')))
        if record['status'] == 'error':
            retry_markets[market_id] = market
            retry_log.write(json.dumps({'event': 'error', 'id': market_id,
                                        'market_ref': market,
                                        'detail': record.get('detail')},
                                       separators=(',', ':')) + '\n')
            retry_log.flush()
        elif market_id in retry_markets:
            retry_markets.pop(market_id, None)
            retry_log.write(json.dumps({'event': 'resolved', 'id': market_id},
                                       separators=(',', ':')) + '\n')
            retry_log.flush()
        log.write(json.dumps(record, separators=(',', ':')) + '\n')
        log.flush()
        status = record['status']
        summary[status] = summary.get(status, 0) + 1
        summary['possibly_truncated'] += int(bool(record.get('possibly_truncated')))
        processed += 1
        if index is not None:
            completed_indexes.add(index)
        if status == 'error' and (summary['error'] <= 5 or summary['error'] % 100 == 0):
            print(f"trade error market={market_id} detail={record.get('detail')}", flush=True)

    def save_checkpoint():
        write_trade_checkpoint(checkpoint_path, {
            'resume_version': 2,
            'next_catalogue_index': next_index,
            'catalogue_size': total_markets,
            'updated_at_unix': time.time(),
        })

    def advance_frontier() -> None:
        nonlocal next_index
        while next_index in completed_indexes:
            completed_indexes.remove(next_index)
            next_index += 1

    def report_progress(previous_frontier: int) -> None:
        if next_index // 1000 > previous_frontier // 1000:
            print(f'trades progress={next_index:,}/{total_markets:,} '
                  f'({next_index / max(total_markets, 1):.1%}) {summary}', flush=True)

    with ThreadPoolExecutor(max_workers=workers) as pool, \
            log_path.open('a', encoding='utf-8') as log, \
            retry_events_path.open('a', encoding='utf-8') as retry_log:
        futures = set()
        with catalogue.open(encoding='utf-8') as handle:
            for line in handle:
                if not line.strip():
                    continue
                current_index = catalogue_index
                catalogue_index += 1
                if current_index >= total_markets:
                    break
                if current_index < start_index:
                    continue
                market = json.loads(line)
                future = pool.submit(fetch_trade_file, market, trades_dir, hourly_dir,
                                     trade_limit, keep_raw, dry_run)
                futures.add(future)
                pending_info[future] = (market, current_index, False)
                if len(futures) >= max_in_flight:
                    done, futures = wait(futures, return_when=FIRST_COMPLETED)
                    old_frontier = next_index
                    for future in done:
                        record_result(future, log, retry_log)
                    advance_frontier()
                    save_checkpoint()
                    report_progress(old_frontier)
            for future in futures:
                record_result(future, log, retry_log)
            old_frontier = next_index
            advance_frontier()
            save_checkpoint()
            report_progress(old_frontier)

        # Retry previous failures after forward collection, once per run. A
        # retry failure remains journaled for a later invocation.
        if next_index >= total_markets and retry_markets:
            retry_list = list(retry_markets.values())
            print(f'retrying {len(retry_list):,} previously failed markets once', flush=True)
            retry_futures = set()
            for market in retry_list:
                future = pool.submit(fetch_trade_file, market, trades_dir, hourly_dir,
                                     trade_limit, keep_raw, dry_run)
                retry_futures.add(future)
                pending_info[future] = (market, None, True)
                if len(retry_futures) >= max_in_flight:
                    done, retry_futures = wait(retry_futures, return_when=FIRST_COMPLETED)
                    for future in done:
                        record_result(future, log, retry_log)
            for future in retry_futures:
                record_result(future, log, retry_log)
    print(f'trade pass complete: catalogue row {next_index:,}/{total_markets:,} '
          f'({next_index / total_markets:.1%}); processed this run={processed:,} {summary}; '
          f'pending retries={len(retry_markets):,}', flush=True)

---
# 5. Sport labels  (🆕 method on top of ♻️ unchanged rules)

### 5a. The rules (♻️ UNCHANGED — `closed_sports_calibration.ipynb`)
Two lists of regular expressions: `sport_rules` (keywords in the text) and `slug_prefix_rules` (league / game codes at the start of a slug such as `epl-`, `cfb-`, `lol-`). They are copied unchanged. The **order of the labels matters**: the first label that matches wins — which is also why single-market keyword matching can be wrong (a team list containing "Arizona" is checked before Basketball).

### 5b. The method (🆕)
A market has an *event*, and the event slug/title usually states the sport outright (`nhl-panthers-vs-bruins`, "CFB: Clemson vs. NC State"). So each market is labelled by trying these layers from most to least specific; the first hit wins and is recorded in `sport_source`:

| Layer | Looks at | Rules | `sport_source` |
|---|---|---|---|
| 0 | **event** slug | hand-written corrections (`EVENT_OVERRIDE_RULES`, currently one entry) | `event_override` |
| 1 | **event** slug | league-code prefix rules (`epl-`, `cfb-`, `euro-` …) | `event_slug_prefix` |
| 2 | **event** slug, first token only if it is ≤ 4 characters | keyword rules (`nhl`, `mlb`, `mls`, `ipl`, `wsop` …) | `event_slug_code` |
| 3 | **event** title + slug | keyword rules | `event_text` |
| 4 | market question + slug | keyword rules — *the original method* | `market_text` |
| 5 | market slug | league-code prefix rules — *the original method* | `market_slug_prefix` |
| – | nothing matched | – | `unclassified` (`sport = "Other / unclassified"`) |

Layers 4–5 alone reproduce the old label exactly; it is always kept in **`sport_market_only`**. **`event_sport`** is the label from layers 0–3 (a property of the event), **`sport`** is the final market label (= `event_sport` when the event could be classified, otherwise layers 4–5).

### 5c. Why this order — evidence from a live sample
Measured while developing this notebook, on two live samples (Gamma API, October 2026):

| Sample | Markets | Events | Unclassified, old method | Unclassified, this method | Markets where event label and old label disagree |
|---|---|---|---|---|---|
| **A** – the 4,000 lowest-id closed Sports markets (2023–24) | 4,000 | 1,130 | 10.3 % | **3.6 %** | 83 markets in 32 events |
| **B** – 1,778 markets from six 2-day windows between Feb 2025 and Sep 2026 | 1,778 | 445 | 4.0 % | **3.3 %** | 59 markets in 29 events |

I read the title of **every one of those 61 events** where the two methods disagree: the event-based label was right in all 61. The old market-only label was typically fooled by a team or place name that also sits in another sport's keyword list ("Georgia", "Arizona", "Washington" → American football; "Kings" → Basketball; "Panthers", "Raiders" → American football; "Rangers" → Baseball). Typical corrections: *NCAA March Madness Winner* American football → **Basketball**; *NHL: Panthers vs. Bruins* American football → **Hockey**; *Euro 2024: Spain vs. Georgia* American football → **Soccer**; *MLB: Royals vs. Cardinals* American football → **Baseball**.

The method is not infallible. Two failure modes seen during testing are handled explicitly: the league-code layer only looks at **short** first tokens (≤ 4 characters) because a long first word is often a country ("england-vs-oman-cricket-…"), and the `afc-` prefix of the original list (Asian Football Confederation) wrongly captured NFL conference markets, which `EVENT_OVERRIDE_RULES` corrects. Others will exist in the full catalogue — §10 lists the biggest event-vs-market disagreements and the remaining unclassified markets so they can be fixed by adding a rule. The numbers above were measured on small samples and will differ on your full catalogue; §10 computes the same figures for your own data.

Two ideas were **tested and rejected**: (a) letting the *market* text override the event (it produces the wrong labels listed above), and (b) filling unclassified markets by a majority vote of their event's other markets (it propagated bad labels — e.g. 4×100 m relays became *Soccer*, Caitlin Clark basketball markets *American football*). Residual `Other / unclassified` markets are still best handled by adding a rule to the lists below — §10 shows a sample of them, as the original notebook did.


In [10]:
# ♻️ UNCHANGED - copied character-for-character from closed_sports_calibration.ipynb.
# Sport is inferred from keywords. The ORDER of the labels matters (first label that matches wins).
sport_rules = [
    ('American football', r'\b(?:NFL|CFB|NCAA Football|Super Bowl|college football|pro football draft|quarterback|starting QB|NFL Draft|touchdown|highest scoring game of Week \d+|fantasy points in Week \d+|combined points scored|Army and Air Force combine|USC and Washington combine|Chiefs|Eagles|Cowboys|Giants|Rams|Seahawks|Vikings|Buccaneers|Broncos|Texans|Colts|Patriots|Chargers|Saints|Ravens|Commanders|Raiders|Jaguars|Bears|Packers|49ers|Bills|Dolphins|Jets|Bengals|Browns|Steelers|Titans|Falcons|Panthers|Cardinals|Lions|Penn State|SMU|Baylor|BYU|Kansas|Alabama|Missouri|Arkansas|Mississippi State|Army|Air Force|Washington|Texas|Georgia|Florida|UCF|Notre Dame|Ohio State|Oklahoma State|West Virginia|Arizona|Arizona State|Texas Tech|Kentucky|Iowa State|Nebraska|Illinois|St\. John\'s)\b'),
    ('Basketball', r'\b(?:NBA|WNBA|basketball|EuroLeague|NCAA Basketball|NCAA Tournament|March Madness|Sweet 16|Final Four|conference tournament|UConn|Connecticut|NBA Draft|NCAAB|Timberwolves|Thunder|Knicks|Celtics|Cavaliers|Magic|Lakers|Clippers|Warriors|Suns|Mavericks|Nuggets|Bucks|Bulls|Heat|76ers|Sixers|Hawks|Hornets|Nets|Pistons|Pacers|Wizards|Raptors|Grizzlies|Pelicans|Rockets|Spurs|Trail Blazers|Blazers|Kings|Jazz)\b'),
    ('Basketball', r'\b(?:vs\. .*combine for 1\d{2}\+? points|Houston vs\. Duke)\b'),
    ('Basketball', r'\bDonovan Mitchell\b'),
    ('Basketball', r'\b(?:NIT Championship|Chattanooga vs\. UC Irvine|Iowa advance to the Quarterfinals)\b'),
    ('Basketball', r'\b(?:ACC title|Jayson Tatum|NCAA Championship game in scoring|Duke win the 2026 ACC)\b'),
    ('Basketball', r'\b(?:LaMelo Ball|LeBron James|Celebrity All-Star Game MVP)\b'),
    ('Basketball', r'\b(?:Kevin Durant|Kyrie Irving|Miami and Duke combine)\b'),
    ('Basketball', r'\b(?:Karl-Anthony Towns|Villanova)\b'),
    ('Basketball', r'\b(?:Naismith Men\'s College Player of the Year|Cameron Boozer)\b'),
    ('Basketball', r'\b(?:SGA attempt|Shai Gilgeous-Alexander|free throws in the game)\b'),
    ('American football', r'\b(?:Fantasy Football|rush for \d[,.]?\d* yards|Saquon Barkley|Tyreek Hill|Travis Kelce|J\.J\. McCarthy|Jalen Hurts|Dak Prescott)\b'),
    ('American football', r'\b(?:passing yards|players drafted|top.*players drafted|Scorigami|Big Game - Winning Conference|Ohio St\.|LSU|Oregon|Hawaii|Nevada)\b'),
    ('American football', r'\b(?:Heisman Trophy|Arch Manning|Jaxson Dart|Shedeur Sanders|Big Game)\b|pro-football-championship'),
    ('American football', r'\b(?:Fantasy Wide Receiver|Garrett Wilson|Justin Jefferson|DK Metcalf|D\.K\. Metcalf)\b'),
    ('American football', r'\b(?:play in Week 1|Chris Olave|Shilo Sanders)\b'),
    ('American football', r'\b(?:Aaron Rodgers|Christian McCaffrey|play Week \d|play again by December)\b'),
    ('American football', r'\bTravis Etienne\b'),
    ('American football', r'\b(?:first TD|Ole Miss|Jonathan Gannon|Odell Beckham Jr\.|Quinnen Williams|Breece Hall|Patrick Mahomes)\b'),
    ('American football', r'\b(?:score a TD|Amon Ra St Brown|Amon-Ra St\. Brown)\b'),
    ('American football', r'\b(?:Clemson|Wake Forest)\b'),
    ('Baseball', r'\b(?:MLB|baseball|World Series|Rockies|Rays|Yankees|Dodgers|Red Sox|Cubs|Mets|Braves|Phillies|Guardians|Royals|Pirates|Cardinals|Tigers|Astros|Mariners|Orioles|Twins|Reds|Padres|Giants|Athletics|Blue Jays|White Sox|Diamondbacks|Nationals|Marlins|Rangers|next team be)\b'),
    ('Baseball', r'\brun scored in the first inning\b'),
    ('Hockey', r'\b(?:NHL|hockey|Stanley Cup|KHL|AHL|IIHF)\b'),
    ('Hockey', r'\b(?:Matthew Tkachuk|score 1\+ goals against the Oilers|Edmonton Oilers)\b'),
    ('Hockey', r'\b(?:Memorial Cup|London Knights|Moncton Wildcats)\b'),
    ('Tennis', r'\b(?:tennis|ATP|WTA|Wimbledon|Grand Slam|ITF|French Open|Australian Open|Indian Wells Open|Miami Open|Madrid Open|Davis Cup|Mallorca Championships|Veolia Arizona Open|Draper vs\. Machac|mixed doubles|singles|Swiss Open: .* vs\.|US Open: .* vs\.|Open: .* vs\.)'),
    ('Tennis', r'\b(?:Hangzhou Open|Casper Ruud win the 2025 Madrid Open|Novak Djokovic win the 2025 Miami Open)\b'),
    ('Tennis', r'\b(?:Terra Wortmann Open|Daniil Medvedev)\b'),
    ('Tennis', r'\b(?:Alcaraz win 3:[01]|Taylor Fritz|Jannik Sinner)\b'),
    ('Tennis', r'\bUS Open: .* and .*\b'),
    ('Tennis', r'\b(?:HSBC Championships|Boss Open|Alexander Zverev|Jiri Lehecka)\b'),
    ('Tennis', r'\b(?:Alexander Bublik|Swiss Open)\b'),
    ('Combat sports', r'\b(?:UFC|MMA|boxing|wrestling|Power Slap|WrestleFest|fight end in (?:Round|the Round)|win by KO|knockdowns? during|KO/TKO|vs\. .*\b(?:fight|round|KO|decision))\b'),
    ('Combat sports', r'\b(?:Pavlovich vs\.|Rozenstruik|Paige VanZant|Covington vs\.|Buckley|Nurmagomedov vs\.|Garcia vs\. Haney|Teofimo Lopez|O\'Malley vs\.|Mairon Santos)\b'),
    ('Combat sports', r'\b(?:Pennington vs\.|Peña|Allen vs\. Hernandez|Stoltzfus vs\.|Ruziboev|Anthony Joshua win in Round|Anthony Joshua|Tyson Fury|Topuria|Holloway)\b'),
    ('Combat sports', r'\b(?:Santos vs\. Tate|Lemos vs\.|Lucindo|Green vs\. Ruffy|Pantoja vs\. Asakura|Ziam vs\. Davis|Ryan Garcia vs\. Rolly Romero)\b'),
    ('Combat sports', r'\b(?:Nathan Fletcher vs\.|Caolán Loughran|Silva vs\. Almeida|Hill vs\. Ricci|Emmers vs\. Miranda|win in Round \d)\b'),
    ('Combat sports', r'\b(?:Murzakanov vs\. Ribeiro|Ankalaev vs\. Rakic|Nikita Krylov vs\. Dominick Reyes|Vergil Ortiz Jr\. vs\. Israil Madrimov|Naimov vs\. Ofli|Brunno Ferreira vs\. Armen Petrosyan|Ponzinibbio vs\. Harris|Yair Rodriguez vs\. Patricio Pitbull)\b'),
    ('Combat sports', r'\b(?:fight reach the \d+(?:st|nd|rd|th) round|fight Jake Paul|Silva vs\. Van|Rosas Jr\. vs\. Morales)\b'),
    ('Combat sports', r'\b(?:Rodriguez vs\. Costa|Bonfim vs\. Williams|Christian Leroy Duncan vs\. Andrey Pulyaev|Alex Morono vs\. Carlos Leal|Jake Paul knockout Mike Tyson|knockout Mike Tyson)\b'),
    ('Combat sports', r'\b(?:Medina vs\. Gautier|Emmett vs\. Murphy)\b'),
    ('Combat sports', r'\b(?:Harrison vs\. Vieira|Johnson vs\. Azaitar)\b'),
    ('Combat sports', r'\b(?:Kape vs\. Almabayev|Santos vs\. Lee|Du Plessis vs\. Adesanya|Oezdemir vs\. Ulberg|Kape vs\. Silva|Moreno vs\. Albazi)\b'),
    ('Cricket', r'\b(?:cricket|IPL|International Cricket|Delhi Capitals|Sunrisers Hyderabad)\b'),
    ('Golf', r'\b(?:golf|PGA|LPGA|Masters Tournament|Masters winner.*amateur|highest finishing debutant.*Masters|first round leader|finish in the Top \d+|leader after Round \d|lowest round|shoot the lowest round|shoot the best round|finish as leader|DP World Tour|LIV Golf|Arnold Palmer Invitational|FedEx St\. Jude|Travelers Championship|Rocket Classic|Cognizant Classic|RBC Heritage|RBC Canadian Open|THE CJ CUP Byron Nelson|Tour Championship|Pinnacle Bank Championship|Danish Golf Championship|Mauritius Open|Ryder Cup|DP AfrAsia|John Deere Classic|Valspar Championship|Charles Schwab Challenge|WM Phoenix Open|The Players Championship|Puerto Rico Open|Corales Puntacana|Valero Texas Open|BMW Championship|Memorial Tournament|ISCO Championship|Texas Children\'s Houston Open|Myrtle Beach Classic|Cadillac Championship|Genesis Scottish Open|Genesis Invitational|Cognizant Classic|Charles Schwab|WM Phoenix|Houston Open|Wyndham Championship|Sony Open|AT\&T Pebble Beach Pro-Am|Biltmore Championship|The American Express tournament|The Open Championship|(?:US|U\.S\.) Open.*(?:finish|Top \d+|leader|lowest round|best round|win)|(?:leader|lowest|best|round|finish).*(?:US|U\.S\.) Open)\b'),
    ('Golf', r'\b(?:RBC Canadian Open|3M Open|Truist Championship|Farmers Insurance Open|Nordea Open|THE CJ CUP Byron Nelson|(?:The )?Open Championship)\b'),
    ('Golf', r'\b(?:Hero World Challenge|The Masters|win the 20\d{2} Masters)\b'),
    ('Golf', r'\b(?:Hole in 1 Challenge|complete the Hole-in-1 Challenge)\b'),
    ('Golf', r'\b(?:hole in one|Presidents Cup|another golfer score the most points|Masters winner be a LIV player)\b'),
    ('Golf', r'\b(?:LIV Greenbrier|LIV Golf)\b'),
    ('Golf', r'\b(?:LIV Riyadh Tournament|LIV.*Tournament)\b'),
    ('Golf', r'\b(?:World Wide Technology Championship|Steven Fisk)\b'),
    ('Motorsports', r'\b(?:F1|Formula 1|NASCAR|MotoGP|IndyCar|Formula E|Grand Prix|Canadian GP|Italy GP|Indy 500|Indianapolis 500|Bommarito Automotive Group 500|Great American Getaway 400|pole|24 Hours of Le Mans|Verstappen|Leclerc|Lewis Hamilton|Bortoleto|DeFrancesco|Hauger|Castroneves|Andretti|Armstrong)\b'),
    ('Motorsports', r'\b(?:Constructors.? Championship|Hamilton vs\. Russell|Mercedes finish second in the 20\d{2})\b'),
    ('Motorsports', r'\b(?:highest Constructor score|Haas have the highest Constructor score|McLaren have the highest Constructor score)\b'),
    ('Motorsports', r'\b(?:Drivers Championship|another driver finish second)\b'),
    ('Motorsports', r'\b(?:Alonso vs\. Stroll|Fernando Alonso|Lance Stroll)\b'),
    ('Motorsports', r'\b(?:Tsunoda vs\. Hadjar|Yuki Tsunoda|Isack Hadjar)\b'),
    ('Cycling', r'\b(?:cycling|Tour de France|Tour of Flanders|Tour de Flanders|Giro d’Italia|Vuelta a España|UCI|BMX)\b'),
    ('Horse racing', r'\b(?:horse racing|Kentucky Derby|Preakness|Belmont Stakes|Queen Anne Stakes|Queen Elizabeth II Jubilee Stakes|Grand National|Jockey Club Gold Cup|Prince of Wales Stakes|WSOP main event)\b'),
    ('Horse racing', r'\bBreeders.? Cup Classic\b'),
    ('Horse racing', r'\b(?:Ascot Gold Cup|Ascot Stakes)\b'),
    ('Horse racing', r'\bTownsville Cup\b'),
    ('Swimming', r'\b(?:swimming|butterfly|freestyle|backstroke|breaststroke|medley|World Aquatics)\b'),
    ('Athletics', r'\b(?:athletics|track and field|marathon|hurdles|high jump|pole vault|long jump|triple jump|shot put|hammer throw|javelin|discus|decathlon|heptathlon|\d{2,4}m [MW])\b'),
    ('Athletics', r'\b(?:gold in \d{2,4}m)\b'),
    ('Athletics', r'\b(?:run the 5k|5k in under|5K race)\b'),
    ('Gymnastics', r'\b(?:gymnastics|vault final|uneven bars?|balance beam|floor exercise)\b'),
    ('Rugby', r'\b(?:rugby|Six Nations|Super Rugby)\b'),
    ('Handball', r'\b(?:handball|EHF)\b'),
    ('Volleyball', r'\b(?:volleyball|FIVB)\b'),
    ('Esports', r'\b(?:esports|League of Legends|Dota 2|Counter-Strike|Valorant|CS2|StarCraft|Rainbow Six|Call of Duty|Honor of Kings|Overwatch|Rocket League|Mobile Legends|MLBB|LEC|MPL|NACL|LCK|LPL|LCS|CBLOL|VCT|EWC|ESL|LOL Worlds|DreamLeague|PGL Wallachia|HLTV|BLAST|MSI|Team Liquid|Elisa Masters|BLAST\.tv|EMEA Masters|Masters Santiago|match point finals|IEM|FRAG Midwest|Blast Premier|BLAST Slam|FalleN|TYLOO|The MongolZ|G2)\b'),
    ('Esports', r'\b(?:PGL|First Stand|LCP|MOUZ|Deep Cross Gaming|Caps win MVP|Kiin win MVP|KSCERATO|Ruler win MVP)\b'),
    ('Esports', r'\b(?:GeoGuessr World Championship|duWapcs get exposed for cheating)\b'),
    ('Esports', r'\b(?:LoL:|GEN vs\. T1|Gen\.G|T1)\b'),
    ('Esports', r'\b(?:Intel Extreme Masters|Team Vitality|Valve add Cache to the Map Pool)\b'),
    ('Esports', r'\bTyler 1 All Star tournament\b'),
    ('Esports', r'\b(?:DreamHack Major|Toronto KOI)\b'),
    ('Esports', r'\b(?:CS Asia Championships|MattCS get exposed for cheating|Legacy win CS)\b'),
    ('Esports', r'\b(?:Astralis|roster change before July)\b'),
    ('Chess', r'\b(?:chess|FIDE|Title Tuesday|World Rapid Championship|World Blitz Championship|Carlsen vs\. Abdusattorov)\b'),
    ('Chess', r'\bSinquefield Cup\b'),
    ('Darts', r'\b(?:darts|PDC World Darts Championship)\b'),
    ('Rodeo', r'\b(?:rodeo|bull riding)\b'),
    ('Competitive eating', r'\b(?:Nathan\'s Famous Hot Dog Eating Contest|hot dog eating contest)\b'),
    ('Competitive eating', r'\b(?:eat the most hot dogs|eating contest)\b'),
    ('Competitive eating', r'\b(?:Joey Chestnut|Nathan\'s Famous Hot Dog 2025)\b'),
    ('Lacrosse', r'\b(?:lacrosse|PLL)\b'),
    ('Pickleball', r'\b(?:pickleball|Major League Pickleball)\b'),
    ('Pickleball', r'\b(?:PPA: Indoor National Championships|PPA Indoor|Federico Staksrud)\b'),
    ('Poker', r'\b(?:poker|WSOP)\b'),
    ('Poker', r'\b(?:Hustler Casino Live|Daniel Negreanu Profit)\b'),
    ('Poker', r'\bHustler Casino\b'),
    ('Action sports', r'\b(?:X Games|skateboard|skateboarding|BMX Freestyle)\b'),
    ('Olympics', r'\b(?:Olympics|Olympic Games|Winter Games|Paralympics|Paralympic Games)\b'),
    ('Olympics', r'\b(?:most medals|3rd most medals)\b'),
    ('Soccer', r'\b(?:soccer|FIFA|UEFA|EPL|UCL|UEL|NWSL|Premier League|La Liga|Bundesliga|Serie A|MLS|Eredivisie|Champions League|Europa League|Conference League|Copa del Rey|Carabao Cup|Club World Cup|football association|starting (?:11|XI)|Anytime Goalscorer|Both Teams to Score|draw at halftime|end in a draw|end in extra time|the match be a draw|first goal|top scorer|90 min|2024 Euros|Euros|transfer window|Manchester United|Manchester City|Real Madrid|Barcelona|Bayern Munich|Borussia Dortmund|Bayer Leverkusen|AC Milan|Inter Milan|Atalanta|Juventus|PSG|Paris Saint-Germain|Liverpool|Arsenal|Chelsea|Tottenham|Newcastle United|Aston Villa|Crystal Palace|Brentford|Wolverhampton|Brighton|Everton|Fulham|Nottingham Forest|West Ham|LOSC Lille|Lille|Feyenoord|Lazio|Girona|Ferencvaros|Fenerbahce|RB Salzburg|Red Star Belgrade|Young Boys|Dinamo Zagreb|Slovan Bratislava|PSV Eindhoven|Stade Brestois|River Plate|Fluminense|Malmo FF|Rangers|Stuttgart|Nice|Leicester City|Southampton|Bournemouth|St Louis City SC|Portland Timbers|FK RFS|FCSB|Atletico Madrid|Atlético Madrid|Real Sociedad|Shakhtar Donetsk|Bologna|Union St-Gilloise|Twente|Celtic|RB Leipzig|Spain vs\.|Portugal|Poland|France|England|Netherlands|Uruguay|Peru|Suriname|Dominican Republic|Bosnia|San Marino|Bolivia|Honduras|El Salvador|Brazil|Japan|Switzerland)\b|\bFC\b'),
    ('Soccer', r'\b(?:Bodo Glimt|Union Saint-Gilloise|Sporting Kansas City|Seattle Sounders|AZ Alkmaar|Ludogorets|Ajax|Qarabag|Club Brugge|Sporting CP|Dynamo Kyiv|Roma|Scotland|Croatia|Slovakia|Chile|Finland U21|Denmark U21|Celtic|RB Leipzig)\b'),
    ('Soccer', r'\b(?:CONCACAF Gold Cup|Inter Miami|Nashville SC|Real Salt Lake|D\.C\. United|Midtjylland|Hoffenheim|Atletico de San Luis|AS Saint-Etienne|Cristiano Ronaldo|Marcus Rashford|Vinicius Junior|Adam Wharton|Bruno Fernandes|Mexico win the 2025 CONCACAF)\b'),
    ('Soccer', r'\b(?:Viktoria Plzen|PAOK|RFS|Maccabi TLV|Slavia Praha|Athletic Bilbao|Ballon d\'Or|win \d-\d)\b'),
    ('Soccer', r'\b(?:Wales beat Iceland|Saudi Arabia beat Haiti|Vancouver Whitecaps|Columbus Crew|LAFC|Colorado Rapids|Benfica|Auckland City|Cade Cowell|score a goal in the 2026 Sidemen Charity Match)\b'),
    ('Soccer', r'\b(?:Ipswich Town and Newcastle|Ipswich Town|Belgium win|Hungary win|Haiti beat Saudi Arabia)\b'),
    ('Soccer', r'\b(?:Empoli|Fiorentina|Braga|IF Elfsborg|Lionel Messi|Lamine Yamal|Ballon d)\b'),
    ('Soccer', r'\b(?:Dortmund|Bayern|Mohamed Salah|Mohammed Salah|Mexico beat USA in Penalties)\b'),
    ('Soccer', r'\b(?:CAF Africa Cup of Nations|Sidemen Charity Match|official 2026 World Cup squad|Frankfurt beat Lyon)\b'),
    ('Soccer', r'\b(?:Euro 2024|Player of the Tournament|Argentina win the World Cup)\b'),
    ('Soccer', r'\b(?:Belgium beat Italy|Trinidad and Tobago beat USA|Guadeloupe beat Jamaica|Italy U21 beat Spain U21)\b'),
    ('Soccer', r'\b(?:Lyon beat Besiktas|Palmeiras beat Al Ahly|Romania U21 beat Spain U21|Mexico win \d[–-]\d|[A-Z][a-z]+ U21 beat [A-Z][a-z]+ U21)\b'),
    ('Soccer', r'\b(?:Lyon|Besiktas|Palmeiras|Al Ahly|Pafos|Flamengo|Esperance Sportive de Tunis|3\+ Goals Scored|World Cup Final: Weather Delay)\b'),
    ('Soccer', r'\b(?:LOSC|Marseille)\b'),
    ('Soccer', r'\b(?:World Cup: Largest Margin of Victory|Lithuania beat Malta)\b'),
    ('Australian rules football', r'\b(?:AFL|Fremantle Dockers|Port Adelaide Power)\b'),
    ('Climbing', r'\b(?:Alex Honnold|free solo Taipei 101)\b'),
    ('Soccer', r'\b(?:CONCACAF Gold Cup|Inter Miami|Nashville SC|Real Salt Lake|D\.C\. United|Midtjylland|Hoffenheim|Atletico de San Luis|AS Saint-Etienne|Cristiano Ronaldo|Marcus Rashford|Vinicius Junior|Adam Wharton|Bruno Fernandes)\b'),
]

# ♻️ UNCHANGED - league/game codes that Polymarket puts at the start of slugs (also from the calibration notebook).
slug_prefix_rules = [
    ('Soccer', r'^(?:fifwc|lal|arg|sea|elc|bra|uel|fif|mex|unl|fl1|bun|es2|clf|chi|kor|bra2|ere|spl|por|nor|swe|tur|bl2|jap|itsb|sud|enl|nwsl|col|col1|fr2|argpn|ja2|rus|lib|den|j2100|uwcl|ukr1|rou1|per1|scop|el2|efl|el1|egy1|efa|conl|uslc|mar1|ned2|chi1|swe2|j1100|ecu1|cze1|gtm|pol|bol1|bel1|kor2|brco|bul|tur2|sui|nor2|uzb1|col2|hr1|ven1|uru1|aut|isl1|itc|dfb|chl2|kaz1|u20wwc|svk1|srb|saf1|irl1|bra3|isr|hun|lva1|est1|acle|usl1|gre1|ltu1|grc|idn1|wsl|uae1|slo|isp|nlc|aze1|aze2|bel2|uef|wwcquefa|qat1|cde|nirl1|skc|sclc|canpl|cdr|asean|argcopa|fro1|caf|idn2|acn|chfa|scoc|cof|cafcl|usc|gsc|usoc|frtc|ptsc|icwq|fin1|ptc|fpd|chi2|atc|ccup|ccc|ucol|auc|trsk|canch|afc|epl|ucl|tha1|aus|geo1|ire|ssc|tpe1|aseanc|concacaf|euro|copa|psg|fa|cl)-'),
    ('Soccer', r'^(?:con)-'),
    ('Esports', r'^(?:lol|dota2|cs2|val|sc2|r6siege|codmw|ow|rl|lec|hok|mlbb|blasttv|ecs|iem|pgl|msi|fissure|blast|the)-'),
    ('Basketball', r'^(?:cbb|cwbb|ncaab|ucol|march|bk[a-z0-9]+)-'),
    ('American football', r'^(?:cfb)-'),
    ('Baseball', r'^(?:npb|kbo|cpbl|lmb|alds|nlds)-'),
    ('Hockey', r'^(?:ahl|khl|shl|dehl|cehl|snhl|mwoh|wwoh|wch)-'),
    ('Golf', r'^(?:dpwt|liv|hsbc|kft)-'),
    ('Tennis', r'^(?:itf|french|australian|internazionali|terra)-'),
    ('Cricket', r'^(?:crban|crint|cricipl|cric[a-z0-9]*)-'),
    ('Combat sports', r'^(?:powerslap|wrestlefest)-'),
    ('Chess', r'^(?:fide)-'),
    ('Lacrosse', r'^(?:pll|wll)-'),
]

In [11]:
# 🆕 The labeller (the rules above are ♻️ unchanged; the layering below is new).
OTHER = 'Other / unclassified'      # ♻️ same literal the calibration notebook used

# ♻️ Compile the rules exactly the way the calibration notebook applied them:
#   * every rule of one label is OR-ed into a single case-insensitive pattern,
#   * labels are tried in order of their FIRST appearance in `sport_rules`, and the first label that matches wins
#     (so the order of labels matters - this is why e.g. "Arizona" hits American football before Basketball).
_combined_sport_rules = {}
for _label, _pattern in sport_rules:
    _combined_sport_rules.setdefault(_label, []).append(f'(?:{_pattern})')
TEXT_RULES = [(label, re.compile('|'.join(patterns), re.IGNORECASE))
              for label, patterns in _combined_sport_rules.items()]
PREFIX_RULES = [(label, re.compile(pattern, re.IGNORECASE)) for label, pattern in slug_prefix_rules]


def _first_match(rules, text: str):
    for label, regex in rules:
        if regex.search(text):
            return label
    return None


def label_market_only(question, slug) -> tuple[str, str]:
    """♻️ The ORIGINAL method of closed_sports_calibration.ipynb (market question + market slug only).

    Kept as its own function so the old labels stay reproducible - they are stored as `sport_market_only`.
    """
    question, slug = str(question or ''), str(slug or '')
    label = _first_match(TEXT_RULES, f'{question} {slug}')            # keywords in question + slug
    if label:
        return label, 'market_text'
    label = _first_match(PREFIX_RULES, slug)                           # league code at the start of the slug
    if label:
        return label, 'market_slug_prefix'
    return OTHER, 'unclassified'


# 🆕 Hand-written corrections for systematic mistakes of the (unchanged) rule lists. Tried FIRST, on the event slug.
# Add an entry here when the disagreement audit in section 10 shows a pattern; the original lists stay untouched.
EVENT_OVERRIDE_RULES = [
    # NFL division / conference markets ("afc-east-champion", "nfc-north-winner", ...). The unchanged
    # slug_prefix_rules file `afc-` under Soccer (Asian Football Confederation), which mislabelled them as Soccer.
    ('American football', r'^(?:afc|nfc)-(?:east|west|north|south|champion)'),
]
OVERRIDE_RULES = [(label, re.compile(pattern, re.IGNORECASE)) for label, pattern in EVENT_OVERRIDE_RULES]


def label_event(title, slug) -> tuple[str, str]:
    """🆕 Label an EVENT from its own slug and title, using the same rule lists as the original method.

    Why event first: the keyword rules are checked label by label, so a text that mentions two sports' words is
    decided by list order, not by meaning - e.g. "Euro 2024: Spain vs. Georgia" -> American football ("Georgia"),
    "Minnesota United FC vs. Sporting Kansas City" -> American football ("Kansas"), "NHL: Kings vs. Panthers"
    -> Basketball ("Kings"). An event slug starts with the league / competition code, which is far more specific
    than any team or place name, so the layers go from most to least specific:
      0. hand-written overrides on the slug                      -> 'event_override'
      1. league-code prefix rules on the slug ("epl-", "cfb-")   -> 'event_slug_prefix'
      2. keyword rules on the slug's first token if it is a short code ("nhl", "mlb", "mls", "ipl") -> 'event_slug_code'
         (<= 4 characters on purpose: longer first words are often countries/teams, e.g. "england-vs-oman-cricket...")
      3. keyword rules on the whole title + slug                 -> 'event_text'
    """
    title, slug = str(title or ''), str(slug or '')
    label = _first_match(OVERRIDE_RULES, slug)
    if label:
        return label, 'event_override'
    label = _first_match(PREFIX_RULES, slug)
    if label:
        return label, 'event_slug_prefix'
    head = re.split(r'[-_]', slug, maxsplit=1)[0]
    if head and len(head) <= 4:
        label = _first_match(TEXT_RULES, head)
        if label:
            return label, 'event_slug_code'
    label = _first_match(TEXT_RULES, f'{title} {slug}')
    if label:
        return label, 'event_text'
    return OTHER, 'unclassified'


def label_market(question, slug, event_sport: str, event_source: str) -> dict:
    """🆕 Final label of a market = its event's label if the event could be classified, otherwise the original
    market-level method. `sport_source` records which layer decided, `sport_market_only` always keeps the old label.
    """
    market_label, market_source = label_market_only(question, slug)
    if event_sport != OTHER:
        sport, source = event_sport, event_source
    else:
        sport, source = market_label, market_source
    return {'sport': sport, 'sport_source': source, 'sport_market_only': market_label}


In [12]:
# 🆕 Smoke test of the layers with real examples (all of these come from the live catalogue).
# `assert` makes this cell fail loudly if someone edits the rules and breaks an expectation.
examples = [
    # (question, market slug, event title, event slug, expected sport, expected layer)
    ('Will Arizona win the NCAA tournament?', 'will-arizona-win-the-ncaa-tournament',
     'NCAA March Madness Winner', 'march-madness-winner', 'Basketball', 'event_slug_prefix'),
    #   ^ old method said "American football" ("Arizona" is in its football team list)
    ('Will Spain Win?', 'will-spain-win-jun-30',
     'Euro 2024: Spain vs. Georgia', 'euro-2024-spain-vs-georgia', 'Soccer', 'event_slug_prefix'),
    #   ^ old method: unclassified (no sport word in the market text); keyword rules on the event title alone
    #     would even say American football because of "Georgia" - this is why the slug prefix is tried first
    ('Los Angeles Kings win the 2024 Stanley Cup?', 'los-angeles-kings-win-the-2024-stanley-cup',
     'NHL Champion', 'nhl-champion-2024', 'Hockey', 'event_slug_code'),
    #   ^ old method said "Basketball" (Sacramento Kings)
    ('NHL: Panthers vs. Bruins', 'nhl-panthers-vs-bruins',
     'NHL: Panthers vs. Bruins', 'nhl-panthers-vs-bruins', 'Hockey', 'event_slug_code'),
    #   ^ without the league-code layer the title is decided by list order ("Panthers" -> American football)
    ('Will Minnesota United FC win on 2025-10-04?', 'mls-min-skc-2025-10-04-min',
     'Minnesota United FC vs. Sporting Kansas City', 'mls-min-skc-2025-10-04', 'Soccer', 'event_slug_code'),
    #   ^ the title alone says American football ("Kansas" is checked before "FC")
    ('Will the Chiefs win the AFC Championship?', 'will-the-chiefs-win-the-afc-championship',
     'AFC Champion', 'afc-champion', 'American football', 'event_override'),
    #   ^ the unchanged prefix list files "afc-" under Soccer; the override corrects NFL conference markets
    ('Will Manchester City beat West Ham?', 'will-manchester-city-beat-west-ham',
     'EPL: West Ham vs. Manchester City', 'epl-west-ham-vs-manchester-city', 'Soccer', 'event_slug_prefix'),
    ('Will Manchester City beat West Ham?', 'will-manchester-city-beat-west-ham',
     '', '', 'Soccer', 'market_text'),                                  # same market, no event info -> falls back to the original method
    ('Will LeBron retire before July?', 'will-lebron-retire-before-july',
     'Will LeBron retire before July?', 'will-lebron-retire-before-july', 'Other / unclassified', 'unclassified'),
    #   ^ a known gap: "LeBron" alone is not in the rules (only "LeBron James"). Such cases show up in the
    #     residual audit in section 10 - the fix is to add a rule, not to guess.
]
for question, market_slug, event_title, event_slug, want_sport, want_source in examples:
    event_sport, event_source = label_event(event_title, event_slug)
    result = label_market(question, market_slug, event_sport, event_source)
    print(f"{question[:48]:<48} -> {result['sport']:<18} via {result['sport_source']:<18} "
          f"(old method: {result['sport_market_only']})")
    assert (result['sport'], result['sport_source']) == (want_sport, want_source), (question, result)
print('labeler examples OK')


Will Arizona win the NCAA tournament?            -> Basketball         via event_slug_prefix  (old method: American football)
Will Spain Win?                                  -> Soccer             via event_slug_prefix  (old method: Other / unclassified)
Los Angeles Kings win the 2024 Stanley Cup?      -> Hockey             via event_slug_code    (old method: Basketball)
NHL: Panthers vs. Bruins                         -> Hockey             via event_slug_code    (old method: American football)
Will Minnesota United FC win on 2025-10-04?      -> Soccer             via event_slug_code    (old method: Soccer)
Will the Chiefs win the AFC Championship?        -> American football  via event_override     (old method: American football)
Will Manchester City beat West Ham?              -> Soccer             via event_slug_prefix  (old method: Soccer)
Will Manchester City beat West Ham?              -> Soccer             via market_text        (old method: Soccer)
Will LeBron retire before Jul

---
# 6. Stage 3 — Master CSV: one row per market  (✏️ MODIFIED)

Grain: **one row per catalogued market**. Each of the 24 hourly bins before close becomes a group of columns (`hour_01_…` = the last hour); each metric cell is a JSON list with one value per outcome token, aligned with `hour_NN_outcomes`.

New (🆕) columns, placed right after the original market columns:

| Column(s) | Meaning |
|---|---|
| `event_id` | id of the market's primary event (the first one Polymarket lists) — join key to `events.jsonl` / the events table |
| `event_ids`, `event_count` | all event ids as a JSON list, and how many (in practice 1) |
| `event_ticker`, `event_slug`, `event_title`, `event_description`, … | the curated event fields from `EVENT_CSV_FIELDS` |
| `event_sport` | sport of the **event** (layers 1–2) |
| `sport` | final sport of the **market** |
| `sport_source` | which layer decided `sport` |
| `sport_market_only` | the original market-only label |

Volume and liquidity are final catalogue snapshots; hourly notional is a trading-activity proxy, not historical market volume (as before).


In [13]:
# ♻️ UNCHANGED - copied from export_all_closed_sports_csv.py
HOURLY_METRICS = [
    "trade_records",
    "unique_transaction_hashes",
    "unique_wallets_observed",
    "buy_records",
    "sell_records",
    "share_size",
    "trade_side_notional_usdc",
    "vwap_price",
    "median_price",
]


def csv_value(value: Any) -> Any:
    """Keep scalar values native to CSV; encode nested source values as JSON."""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False, separators=(",", ":"))
    if value is None:
        return ""
    if isinstance(value, bool):
        return "true" if value else "false"
    return value


def discover_market_columns(catalogue: Path) -> tuple[list[str], int]:
    columns: set[str] = set()
    count = 0
    with catalogue.open("r", encoding="utf-8") as source:
        for line_number, line in enumerate(source, 1):
            if not line.strip():
                continue
            try:
                market = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(f"Invalid JSON in {catalogue}, line {line_number}: {exc}") from exc
            if not isinstance(market, dict):
                raise RuntimeError(f"Expected a JSON object in {catalogue}, line {line_number}")
            columns.update(market.keys())
            count += 1
    if not count:
        raise RuntimeError(f"No market records found in {catalogue}")
    return sorted(columns), count

In [14]:
# 🆕 Event lookup + enrichment used by the export
def single_line(value):
    """🆕 Event descriptions contain line breaks. In the CSVs they become spaces so that every row stays on ONE
    line (safe for `wc -l`, grep, spreadsheet previews). The untouched text is still in events.jsonl."""
    return value.replace('\r\n', ' ').replace('\n', ' ').replace('\r', ' ') if isinstance(value, str) else value


def flatten_event(event: dict) -> tuple[dict, str, str]:
    """🆕 One event -> (CSV-ready `event_*` columns, event_sport, event_sport_source)."""
    columns = {f'event_{field}': csv_value(single_line(event.get(field))) for field in EVENT_CSV_FIELDS}
    event_sport, event_source = label_event(event.get('title'), event.get('slug'))
    return columns, event_sport, event_source


class EventStore:
    """🆕 Random-access reader for events.jsonl.

    The master CSV needs the event of every market. Loading every event (with its long description) into memory
    would cost gigabytes for a large catalogue, so only a {event_id: byte offset} index is kept in memory; the
    event itself is read from disk on demand, with a small LRU cache because the markets of one event are
    neighbours in the catalogue (consecutive ids).
    """

    def __init__(self, path: Path, cache_size: int = 20_000):
        self.offsets: dict[str, int] = {}
        with path.open('rb') as handle:
            offset = 0
            for line in handle:
                if line.strip():
                    try:
                        self.offsets.setdefault(str(json.loads(line)['id']), offset)
                    except (KeyError, json.JSONDecodeError):
                        pass
                offset += len(line)
        self._handle = path.open('rb')
        self.row = lru_cache(maxsize=cache_size)(self._row)
        # What a market without any event gets: empty event columns, event_sport = unclassified.
        self.blank = ({f'event_{field}': '' for field in EVENT_CSV_FIELDS}, OTHER, 'unclassified')

    def _row(self, event_id: str) -> tuple[dict, str, str]:
        offset = self.offsets.get(event_id)
        if offset is None:                      # link points to an event that is not in events.jsonl
            return self.blank
        self._handle.seek(offset)
        return flatten_event(json.loads(self._handle.readline()))

    def close(self):
        self._handle.close()


def load_market_event_links(path: Path) -> dict[str, str]:
    """🆕 {market_id: "event_id[,event_id...]"} from market_events.jsonl (the first id is the primary event)."""
    links: dict[str, str] = {}
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                try:
                    record = json.loads(line)
                    links.setdefault(str(record['id']), ','.join(record.get('event_ids') or []))
                except (KeyError, json.JSONDecodeError):
                    continue
    return links


def enrich_market(market: dict, event_id_text: str | None, store: EventStore) -> dict[str, Any]:
    """🆕 All the columns this notebook adds to a market row: event info + sport labels."""
    event_ids = event_id_text.split(',') if event_id_text else []
    columns, event_sport, event_source = store.row(event_ids[0]) if event_ids else store.blank
    row = {'event_id': event_ids[0] if event_ids else '',      # primary event = first one listed by Polymarket
           'event_ids': json.dumps(event_ids) if event_ids else '',
           'event_count': len(event_ids)}
    row.update(columns)
    row['event_sport'] = event_sport
    row.update(label_market(market.get('question'), market.get('slug'), event_sport, event_source))
    return row


In [15]:
def export_master_csv(data_dir: Path = DATA_DIR, output: Path | None = None, force: bool = False,
                      progress_every: int = 100_000) -> Path:
    """Export one row per closed-sports market with the 24 hourly bins as columns.

    ✏️ MODIFIED from export_all_closed_sports_csv.main(): argparse -> function arguments, parser.error ->
    exceptions, and the 🆕 event + sport columns. Every line that is not marked 🆕 is the original logic.
    """
    data_dir = Path(data_dir).expanduser().resolve()
    catalogue = data_dir / "markets.jsonl"
    hourly_dir = data_dir / "hourly_activity_24h"
    events_path = data_dir / "events.jsonl"                    # 🆕
    links_path = data_dir / "market_events.jsonl"              # 🆕
    output = (Path(output).expanduser().resolve() if output else
              data_dir / "closed_sports_all_data.csv")
    if not catalogue.is_file():
        raise FileNotFoundError(f"catalogue not found: {catalogue}")
    if not hourly_dir.is_dir():
        raise FileNotFoundError(f"hourly aggregate directory not found: {hourly_dir}")
    for path in (events_path, links_path):                     # 🆕
        if not path.is_file():
            raise FileNotFoundError(f"{path.name} not found in {data_dir}: run the discovery stage first "
                                    "(it now also saves the events and backfills older catalogues).")
    if output.exists() and not force:
        raise FileExistsError(f"output already exists (set FORCE_EXPORT = True to replace): {output}")
    if progress_every < 1:
        raise ValueError("progress_every must be at least 1")

    print(f"scanning catalogue fields: {catalogue}", flush=True)
    market_fields, total = discover_market_columns(catalogue)
    preferred_market_fields = [
        "id", "conditionId", "slug", "question", "startDate", "endDate", "closedTime",
        "outcomes", "outcomePrices", "clobTokenIds", "volume", "volumeNum", "volumeClob",
        "volumeAmm", "liquidity", "liquidityNum", "liquidityClob", "liquidityAmm", "spread",
        "bestBid", "bestAsk", "lastTradePrice",
    ]
    market_order = [field for field in preferred_market_fields if field in market_fields]
    market_order += [field for field in market_fields if field not in market_order]
    # 🆕 The new columns sit right after the original market columns, so the original columns keep their
    # relative order and the hourly block still follows them.
    fields = market_order + EVENT_COLUMNS + SPORT_COLUMNS + ["hourly_file_exists", "has_hourly_activity", "hourly_read_error"]
    for hour in range(1, 25):
        prefix = f"hour_{hour:02d}"
        fields.extend([f"{prefix}_start_utc", f"{prefix}_end_utc", f"{prefix}_outcomes"])
        fields.extend(f"{prefix}_{metric}" for metric in HOURLY_METRICS)
    print(f"catalogue markets={total:,}; market columns={len(market_fields)}; "
          f"new event/sport columns={len(EVENT_COLUMNS) + len(SPORT_COLUMNS)}; "
          f"hourly columns={24 * (3 + len(HOURLY_METRICS))}; output={output}", flush=True)

    print("indexing events and market->event links ...", flush=True)       # 🆕
    store = EventStore(events_path)
    market_to_events = load_market_event_links(links_path)
    without_event = 0

    output.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{output.name}.", suffix=".partial", dir=output.parent)
    os.close(fd)
    temp_path = Path(temp_name)
    processed = 0
    rows_written = 0
    with_hourly = 0
    files_found = 0
    read_errors = 0
    try:
        with catalogue.open("r", encoding="utf-8") as source, temp_path.open(
            "w", encoding="utf-8", newline="", buffering=1024 * 1024
        ) as target:
            writer = csv.DictWriter(target, fieldnames=fields, extrasaction="ignore", lineterminator="\n")
            writer.writeheader()
            for line_number, line in enumerate(source, 1):
                if not line.strip():
                    continue
                try:
                    market = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise RuntimeError(f"Invalid JSON in {catalogue}, line {line_number}: {exc}") from exc
                market_id = str(market.get("id", ""))
                activity_path = hourly_dir / f"{market_id}.json.gz"
                file_exists = activity_path.is_file()
                aggregates: list[dict[str, Any]] = []
                read_error = False
                if file_exists:
                    files_found += 1
                    try:
                        with gzip.open(activity_path, "rt", encoding="utf-8") as handle:
                            payload = json.load(handle)
                        if not isinstance(payload, list):
                            raise ValueError("expected a JSON list of hourly aggregates")
                        aggregates = [row for row in payload if isinstance(row, dict)]
                    except (OSError, EOFError, UnicodeDecodeError, json.JSONDecodeError, ValueError) as exc:
                        read_errors += 1
                        read_error = True
                        print(f"warning: market={market_id} hourly file unreadable: {type(exc).__name__}: {exc}",
                              file=sys.stderr, flush=True)

                row = {key: csv_value(value) for key, value in market.items()}
                row.update(enrich_market(market, market_to_events.get(market_id), store))   # 🆕
                without_event += not row["event_id"]                                         # 🆕
                close_time = None
                close_text = market.get("endDate")
                if close_text:
                    try:
                        close_time = datetime.fromisoformat(str(close_text).replace("Z", "+00:00")).astimezone(timezone.utc)
                    except (ValueError, TypeError):
                        close_time = None
                per_hour: dict[int, list[dict[str, Any]]] = {hour: [] for hour in range(1, 25)}
                for aggregate in aggregates:
                    try:
                        hour = int(aggregate["hours_before_close"])
                    except (KeyError, TypeError, ValueError):
                        continue
                    if 1 <= hour <= 24:
                        per_hour[hour].append(aggregate)

                for hour in range(1, 25):
                    prefix = f"hour_{hour:02d}"
                    records = per_hour[hour]
                    starts = {record.get("hour_start_utc") for record in records if record.get("hour_start_utc")}
                    ends = {record.get("hour_end_utc") for record in records if record.get("hour_end_utc")}
                    if close_time is not None:
                        hour_start = (close_time - timedelta(hours=hour)).isoformat()
                        hour_end = (close_time - timedelta(hours=hour - 1)).isoformat()
                    else:
                        hour_start = next(iter(starts), "")
                        hour_end = next(iter(ends), "")
                    row[f"{prefix}_start_utc"] = hour_start
                    row[f"{prefix}_end_utc"] = hour_end
                    identity = [
                        {"asset": record.get("asset", ""), "outcome": record.get("outcome", ""),
                         "outcome_index": record.get("outcome_index")}
                        for record in records
                    ]
                    row[f"{prefix}_outcomes"] = json.dumps(identity, ensure_ascii=False, separators=(",", ":")) if identity else ""
                    for metric in HOURLY_METRICS:
                        # Values align by position with the outcome/token list
                        # above, avoiding repeated long token IDs in every cell.
                        values = [record.get(metric) for record in records]
                        row[f"{prefix}_{metric}"] = json.dumps(values, ensure_ascii=False, separators=(",", ":")) if values else ""

                row["hourly_file_exists"] = "true" if file_exists else "false"
                row["has_hourly_activity"] = "true" if any(per_hour.values()) else "false"
                row["hourly_read_error"] = "true" if read_error else "false"
                if any(per_hour.values()):
                    with_hourly += 1
                writer.writerow(row)
                rows_written += 1

                processed += 1
                if processed % progress_every == 0 or processed == total:
                    target.flush()
                    size_mb = temp_path.stat().st_size / (1024 * 1024)
                    print(f"progress markets={processed:,}/{total:,} ({processed / total:.1%}) "
                          f"rows={rows_written:,} with_hourly_activity={with_hourly:,} "
                          f"hourly_files={files_found:,} read_errors={read_errors:,} size={size_mb:,.0f} MiB",
                          flush=True)

        if output.exists() and not force:
            raise FileExistsError(f"output appeared during export (not replacing): {output}")
        os.replace(temp_path, output)
    except BaseException:
        try:
            temp_path.unlink(missing_ok=True)
        except OSError:
            pass
        raise
    finally:
        store.close()                                                                         # 🆕

    size_gb = output.stat().st_size / (1024 ** 3)
    print(f"complete: {output}", flush=True)
    print(f"markets={processed:,}; rows={rows_written:,}; markets_with_hourly_activity={with_hourly:,}; "
          f"hourly_files={files_found:,}; unreadable_hourly_files={read_errors:,}; size={size_gb:,.2f} GiB",
          flush=True)
    print(f"markets without event info={without_event:,}", flush=True)                          # 🆕
    print("Note: hourly cells contain JSON lists for each outcome/token; individual raw trades were not retained.", flush=True)
    return output


---
# 7. Stage 4 — One forecast per usable settled binary market  (✏️ MODIFIED)

For every settled binary market we take the **latest hour before close that has a trade** and use the VWAP price of the first outcome as the forecast `probability` (or 1 − the second outcome's VWAP if only that token traded in the hour), with `actual` = 1 if the first outcome won (final price ≥ 0.98) and 0 if it lost (≤ 0.02); unsettled or ambiguous markets are dropped. The logic is the original one, applied chunk by chunk so the multi-GB master CSV never has to fit in memory.

🆕 Eight extra columns are appended **after** the 12 original ones: `event_id`, `event_title`, `event_slug`, `event_ticker`, `event_sport`, `sport`, `sport_source`, `sport_market_only`. If you point this at a master CSV from the old script (no event columns), they are simply skipped.


In [16]:
# ♻️ UNCHANGED - copied from prepare_sports_forecasts.py
OUTPUT_COLUMNS = [
    "market_id", "question", "slug", "first_outcome", "probability", "actual",
    "hours_before_close", "latest_hour_trade_records",
    "latest_hour_trade_side_notional_usdc", "volumeNum", "liquidityNum", "spread",
]


def as_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return []
    return []


# 🆕 Columns appended AFTER the 12 original forecast columns (so the original columns are unchanged).
FORECAST_EXTRA_COLUMNS = ['event_id', 'event_title', 'event_slug', 'event_ticker',
                          'event_sport', 'sport', 'sport_source', 'sport_market_only']


In [17]:
# ✏️ MODIFIED from prepare_sports_forecasts.py - only the lines marked 🆕 differ
def forecasts_for_chunk(chunk: pd.DataFrame, extra_columns=()) -> tuple[pd.DataFrame, int]:  # 🆕 extra_columns
    """Use pandas chunk operations to get the nearest-hour binary forecasts."""
    markets = chunk[["id", "question", "slug", "outcomes", "outcomePrices",
                     "volumeNum", "liquidityNum", "spread"] + list(extra_columns)].copy()  # 🆕 carry event/sport columns
    markets["market_id"] = markets["id"].astype("string")
    markets["outcomes"] = markets["outcomes"].map(as_list)
    markets["settled"] = markets["outcomePrices"].map(as_list)
    binary = markets.loc[
        markets["outcomes"].str.len().eq(2) & markets["settled"].str.len().eq(2)
    ].copy()
    binary["first_result"] = pd.to_numeric(binary["settled"].str[0], errors="coerce")
    binary = binary.loc[binary["first_result"].le(0.02) | binary["first_result"].ge(0.98)].copy()
    binary["actual"] = binary["first_result"].ge(0.98).astype("int8")
    binary["first_outcome"] = binary["outcomes"].str[0]
    binary_n = len(binary)
    if binary.empty:
        return pd.DataFrame(columns=OUTPUT_COLUMNS + list(extra_columns)), binary_n  # 🆕

    metadata = binary.set_index("market_id")[[
        "question", "slug", "first_outcome", "actual", "volumeNum", "liquidityNum", "spread"
    ] + list(extra_columns)]  # 🆕
    activity_parts = []
    for hour in range(1, 25):
        prefix = f"hour_{hour:02d}"
        part = pd.DataFrame(index=binary.index)
        part["market_id"] = binary["market_id"]
        part["token"] = chunk.loc[binary.index, f"{prefix}_outcomes"].map(as_list)
        part["vwap_price"] = chunk.loc[binary.index, f"{prefix}_vwap_price"].map(as_list)
        part["trade_records"] = chunk.loc[binary.index, f"{prefix}_trade_records"].map(as_list)
        part["trade_side_notional_usdc"] = chunk.loc[binary.index, f"{prefix}_trade_side_notional_usdc"].map(as_list)
        part = part.explode(["token", "vwap_price", "trade_records", "trade_side_notional_usdc"])
        part = part.loc[part["token"].map(lambda value: isinstance(value, dict))].copy()
        if part.empty:
            continue
        part["outcome_index"] = pd.to_numeric(part["token"].str.get("outcome_index"), errors="coerce")
        part["vwap_price"] = pd.to_numeric(part["vwap_price"], errors="coerce")
        part["trade_records"] = pd.to_numeric(part["trade_records"], errors="coerce")
        part["trade_side_notional_usdc"] = pd.to_numeric(part["trade_side_notional_usdc"], errors="coerce")
        part["hours_before_close"] = hour
        valid = part["outcome_index"].isin([0, 1]) & part["vwap_price"].between(0, 1)
        activity_parts.append(part.loc[valid, [
            "market_id", "hours_before_close", "outcome_index", "vwap_price",
            "trade_records", "trade_side_notional_usdc",
        ]])

    if not activity_parts:
        return pd.DataFrame(columns=OUTPUT_COLUMNS + list(extra_columns)), binary_n  # 🆕
    activity = pd.concat(activity_parts, ignore_index=True)
    nearest_hour = activity.groupby("market_id")["hours_before_close"].transform("min")
    latest = activity.loc[activity["hours_before_close"].eq(nearest_hour)].copy()
    prices = latest.pivot_table(
        index="market_id", columns="outcome_index", values="vwap_price", aggfunc="first"
    ).reindex(columns=[0, 1])
    prices["probability"] = prices[0].fillna(1 - prices[1])
    aggregates = latest.groupby("market_id").agg(
        hours_before_close=("hours_before_close", "min"),
        latest_hour_trade_records=("trade_records", "sum"),
        latest_hour_trade_side_notional_usdc=("trade_side_notional_usdc", "sum"),
    )
    result = prices[["probability"]].join(aggregates).join(metadata)
    result = result.dropna(subset=["probability"]).reset_index()
    return result.reindex(columns=OUTPUT_COLUMNS + list(extra_columns)), binary_n  # 🆕

In [18]:
def build_forecasts(master: Path = MASTER_CSV, output: Path = FORECASTS_CSV, force: bool = False,
                    chunk_size: int = 25_000) -> Path:
    """Build one forecast row per usable settled binary market from the wide master CSV.

    ✏️ MODIFIED from prepare_sports_forecasts.main(): argparse -> arguments, fixed paths -> arguments,
    and the 🆕 extra columns. Every line that is not marked 🆕 is the original logic.
    """
    if not master.is_file():
        raise FileNotFoundError(master)
    if output.exists() and not force and output.stat().st_mtime >= master.stat().st_mtime:
        print(f"Already prepared from the current master CSV: {output}")
        return output

    data_dir = output.parent
    data_dir.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix="closed_sports_forecasts_", suffix=".csv.tmp", dir=data_dir)
    os.close(fd)
    temporary = Path(temporary_name)
    catalogue_n = binary_n = forecast_n = 0
    wrote_header = False
    try:
        usecols = ["id", "question", "slug", "outcomes", "outcomePrices",
                   "volumeNum", "liquidityNum", "spread"]
        # 🆕 only ask for the extra columns the master CSV really has (an older master CSV has none of them)
        header = pd.read_csv(master, nrows=0).columns
        extra_columns = [column for column in FORECAST_EXTRA_COLUMNS if column in header]
        usecols.extend(extra_columns)
        for hour in range(1, 25):
            prefix = f"hour_{hour:02d}"
            usecols.extend([
                f"{prefix}_outcomes", f"{prefix}_vwap_price", f"{prefix}_trade_records",
                f"{prefix}_trade_side_notional_usdc",
            ])
        dtypes = {"id": "string", **{c: "string" for c in ("event_id",) if c in extra_columns}}   # 🆕 ids stay text
        for chunk in pd.read_csv(
            master, usecols=usecols, dtype=dtypes, chunksize=chunk_size, low_memory=False
        ):
            catalogue_n += len(chunk)
            forecasts, settled_binary_n = forecasts_for_chunk(chunk, extra_columns)
            binary_n += settled_binary_n
            if not forecasts.empty:
                forecasts.to_csv(temporary, mode="a", header=not wrote_header, index=False)
                wrote_header = True
                forecast_n += len(forecasts)

            print(
                f"Scanned {catalogue_n:,} sports markets from master CSV; "
                f"settled binary={binary_n:,}; usable forecasts={forecast_n:,}",
                flush=True,
            )

        if not wrote_header:
            raise ValueError("No usable settled Sports forecasts found; CSV was not created.")
        temporary.replace(output)
    finally:
        temporary.unlink(missing_ok=True)

    print(f"Saved {forecast_n:,} one-per-market forecasts to {output}")
    return output


---
# 8. 🆕 Events table: one row per event

Handy when the unit of analysis is the game/event rather than the individual market. Contains the `event_*` columns, the event's sport label (`event_sport`, `event_sport_source`) and `n_markets_in_catalogue` (how many catalogued markets belong to the event).


In [19]:
def export_events_table(data_dir: Path = DATA_DIR, output: Path | None = None, force: bool = False) -> Path:
    """🆕 One row per event: all `event_*` columns, its sport label and how many catalogued markets it has."""
    data_dir = Path(data_dir).expanduser().resolve()
    events_path, links_path = data_dir / 'events.jsonl', data_dir / 'market_events.jsonl'
    output = Path(output).expanduser().resolve() if output else data_dir / 'closed_sports_events.csv'
    for path in (events_path, links_path):
        if not path.is_file():
            raise FileNotFoundError(f'{path.name} not found in {data_dir}: run the discovery stage first.')
    if output.exists() and not force:
        raise FileExistsError(f'output already exists (set FORCE_EXPORT = True to replace): {output}')

    markets_per_event: Counter = Counter()           # markets are counted under every event they belong to
    with links_path.open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                try:
                    markets_per_event.update(json.loads(line).get('event_ids') or [])
                except json.JSONDecodeError:
                    continue

    columns = ['event_id'] + [f'event_{field}' for field in EVENT_CSV_FIELDS] + \
              ['event_sport', 'event_sport_source', 'n_markets_in_catalogue']
    fd, temp_name = tempfile.mkstemp(prefix=f'.{output.name}.', suffix='.partial', dir=output.parent)
    os.close(fd)
    temp_path = Path(temp_name)
    rows = 0
    try:
        with events_path.open(encoding='utf-8') as source, \
                temp_path.open('w', encoding='utf-8', newline='') as target:
            writer = csv.DictWriter(target, fieldnames=columns, extrasaction='ignore', lineterminator='\n')
            writer.writeheader()
            for line in source:
                if not line.strip():
                    continue
                event = json.loads(line)
                flat, event_sport, event_source = flatten_event(event)
                writer.writerow({'event_id': event['id'], **flat, 'event_sport': event_sport,
                                 'event_sport_source': event_source,
                                 'n_markets_in_catalogue': markets_per_event.get(str(event['id']), 0)})
                rows += 1
        os.replace(temp_path, output)
    except BaseException:
        temp_path.unlink(missing_ok=True)
        raise
    print(f'Saved {rows:,} events to {output}')
    return output


---
# 9. Run the pipeline  (🆕)

This single cell is what `python collect_all_closed_sports.py`, `python export_all_closed_sports_csv.py` and `python prepare_sports_forecasts.py` did one after another. Stages are switched on/off in the configuration cell.

> ⚠️ With the default configuration this starts the **full collection** (hours). For a first try set `MAX_PAGES = 3` in §1.


In [20]:
# 🆕 The "main()" of the combined pipeline. Every stage is resumable: if the kernel dies or you stop the cell,
# just run this cell again and it continues where it stopped.
DATA_DIR.mkdir(parents=True, exist_ok=True)
started = time.time()

if RUN_DISCOVERY:
    discover(DATA_DIR, PAGE_SIZE, MAX_PAGES, DRY_RUN)

if RUN_TRADES:
    if DRY_RUN and not (DATA_DIR / 'markets.jsonl').exists():
        print('DRY RUN: trade stage needs an existing catalogue; dry-run discovery does not write one.')
    else:
        collect_trades(DATA_DIR, WORKERS, TRADE_LIMIT, MAX_MARKETS, KEEP_RAW, DRY_RUN)

if not DRY_RUN:
    # The three derived files are rebuilt only on request: they are cheap compared with the trade stage, but the
    # master CSV can be several GB. Set FORCE_EXPORT = True after you have collected more trades / markets.
    if RUN_EXPORT:
        if MASTER_CSV.exists() and not FORCE_EXPORT:
            if set(EVENT_COLUMNS + SPORT_COLUMNS) <= set(pd.read_csv(MASTER_CSV, nrows=0).columns):
                print(f'Master CSV already exists, not rebuilding (set FORCE_EXPORT = True to rebuild): {MASTER_CSV}')
            else:
                # A master CSV written by the OLD export script. It is never overwritten without being asked.
                print(f'⚠️ {MASTER_CSV.name} was built by the old export script: it has no event / sport columns.\n'
                      '   Set FORCE_EXPORT = True in §1 and run this cell again to rebuild it. Every original column\n'
                      '   comes out identical (same names, order and values); the file only gains the new columns.')
        else:
            export_master_csv(DATA_DIR, MASTER_CSV, force=FORCE_EXPORT, progress_every=PROGRESS_EVERY)
    if RUN_FORECASTS:
        build_forecasts(MASTER_CSV, FORECASTS_CSV, force=FORCE_FORECASTS, chunk_size=FORECAST_CHUNK_SIZE)
    if RUN_EVENTS_TABLE:
        if EVENTS_CSV.exists() and not FORCE_EXPORT:
            print(f'Events table already exists, not rebuilding (set FORCE_EXPORT = True to rebuild): {EVENTS_CSV}')
        else:
            export_events_table(DATA_DIR, EVENTS_CSV, force=FORCE_EXPORT)

print(f'\nPipeline finished in {(time.time() - started) / 60:.1f} min')


discovery keyset_page=1 received=100 new=100 total=100 events=30
discovery keyset_page=2 received=100 new=100 total=200 events=52
discovery keyset_page=3 received=100 new=100 total=300 events=77
discovery complete: 300 unique closed Sports markets in /Users/pietrorebecchi/Desktop/Final/data/all_closed_sports/markets.jsonl; 77 events in /Users/pietrorebecchi/Desktop/Final/data/all_closed_sports/events.jsonl
legacy run indexed: resume at catalogue row 0/300; queued 0 unsaved earlier markets for retry
resuming at catalogue row 1/300 (next row checkpointed: 0)
trade pass complete: catalogue row 300/300 (100.0%); processed this run=300 {'cached': 0, 'downloaded': 300, 'error': 0, 'skipped': 0, 'would fetch': 0, 'possibly_truncated': 0}; pending retries=0
scanning catalogue fields: /Users/pietrorebecchi/Desktop/Final/data/all_closed_sports/markets.jsonl
catalogue markets=300; market columns=22; new event/sport columns=36; hourly columns=288; output=/Users/pietrorebecchi/Desktop/Final/data/al

---
# 10. Checks and audit  (🆕, with the residual audit ♻️ from the calibration notebook)

Run after the pipeline: file inventory, integrity checks (no duplicate market ids, how many markets have an event), a breakdown of how sport labels were decided, and a sample of what is still unclassified.


In [21]:
# 🆕 Quick inventory of everything the pipeline has produced so far.
def count_lines(path: Path) -> int | None:
    if not path.exists():
        return None
    with path.open('rb') as handle:
        return sum(1 for line in handle if line.strip())

def count_csv_rows(path: Path) -> int | None:
    if not path.exists():
        return None
    return sum(len(chunk) for chunk in pd.read_csv(path, usecols=[0], chunksize=200_000, dtype=str))

inventory = pd.DataFrame([
    ('markets.jsonl                (catalogue)', count_lines(DATA_DIR / 'markets.jsonl')),
    ('events.jsonl                 (🆕 unique events)', count_lines(DATA_DIR / 'events.jsonl')),
    ('market_events.jsonl          (🆕 market -> event links)', count_lines(DATA_DIR / 'market_events.jsonl')),
    ('hourly_activity_24h/*.json.gz (collected markets)',
     sum(1 for _ in (DATA_DIR / 'hourly_activity_24h').glob('*.json.gz')) if (DATA_DIR / 'hourly_activity_24h').is_dir() else None),
    ('closed_sports_all_data.csv   (master, rows)', count_csv_rows(MASTER_CSV)),
    ('closed_sports_market_forecasts.csv (forecasts)', count_csv_rows(FORECASTS_CSV)),
    ('closed_sports_events.csv     (🆕 events)', count_csv_rows(EVENTS_CSV)),
], columns=['file', 'rows / files'])
display(inventory)


,file,rows / files
0,markets.jsonl (catalogue),300
1,events.jsonl (🆕 unique events),77
2,market_events.jsonl (🆕 market -> even...,300
3,hourly_activity_24h/*.json.gz (collected markets),300
4,"closed_sports_all_data.csv (master, rows)",300
5,closed_sports_market_forecasts.csv (forecasts),57
6,closed_sports_events.csv (🆕 events),77


In [22]:
# 🆕 Integrity checks on the master CSV + how the sport labels were decided.
# Only the few columns we need are loaded (the full file can be several GB).
needed = ['id', 'question', 'slug', 'event_id', 'event_title', 'event_sport', 'sport', 'sport_source', 'sport_market_only']
labels = None
if not MASTER_CSV.exists():
    print('Master CSV not built yet.')
elif not set(needed) <= set(pd.read_csv(MASTER_CSV, nrows=0).columns):
    print('The master CSV has no event / sport columns yet (it was built by the old export script):\n'
          'set FORCE_EXPORT = True in section 1 and run section 9 again.')
else:
    labels = pd.read_csv(MASTER_CSV, usecols=needed, dtype={'id': 'string', 'event_id': 'string'}, low_memory=False)
    print(f'Markets in master CSV: {len(labels):,}')
    print(f'Duplicate market ids:  {labels["id"].duplicated().sum():,}   (must be 0)')
    print(f'Markets without event: {labels["event_id"].isna().sum():,}   unique events: {labels["event_id"].nunique():,}')
    unclassified = labels['sport'].eq(OTHER).sum()
    print(f'Unclassified markets:  {unclassified:,} ({unclassified / len(labels):.1%})   '
          f'[original market-only method: {labels["sport_market_only"].eq(OTHER).mean():.1%}]')
    print('\nWhich layer decided the label:')
    display(labels['sport_source'].value_counts().rename('markets').to_frame())
    print('Markets per sport:')
    display(labels['sport'].value_counts().rename('markets').to_frame())
    changed = labels['sport'].ne(labels['sport_market_only']) & labels['sport_market_only'].ne(OTHER)
    print(f'Markets whose final label DIFFERS from the old market-only label (old label was not "unclassified"): '
          f'{changed.sum():,} ({changed.mean():.1%})')
    # The audit that matters: where the event says one sport and the old market-only method another, one of them is
    # wrong. One row per event, biggest first. Read it: if the EVENT label is the wrong one, fix the rule that produced
    # it (see `sport_source`) or add an entry to EVENT_OVERRIDE_RULES, then re-export.
    disagree = labels['event_sport'].ne(OTHER) & labels['sport_market_only'].ne(OTHER) & \
               labels['event_sport'].ne(labels['sport_market_only'])
    print(f'\nEvent label vs old market-only label DISAGREE for {disagree.sum():,} markets in '
          f'{labels.loc[disagree, "event_id"].nunique():,} events. Largest 30 events:')
    display(labels.loc[disagree].groupby(['event_title', 'event_sport', 'sport_market_only', 'sport_source'], dropna=False)
            .size().rename('markets').sort_values(ascending=False).head(30).reset_index())


Markets in master CSV: 300
Duplicate market ids:  0   (must be 0)
Markets without event: 0   unique events: 77
Unclassified markets:  17 (5.7%)   [original market-only method: 6.0%]

Which layer decided the label:


,markets
sport_source,
event_slug_code,143
event_text,121
unclassified,17
market_text,15
event_slug_prefix,4


Markets per sport:


,markets
sport,
American football,124
Basketball,109
Soccer,23
Other / unclassified,17
Combat sports,10
Esports,5
Motorsports,4
Chess,2
Tennis,2


Markets whose final label DIFFERS from the old market-only label (old label was not "unclassified"): 0 (0.0%)

Event label vs old market-only label DISAGREE for 0 markets in 0 events. Largest 30 events:


,event_title,event_sport,sport_market_only,sport_source,markets


In [23]:
# ♻️ Residual audit - same idea and same random seed as closed_sports_calibration.ipynb: look at what is STILL
# unclassified before adding another rule to `sport_rules` / `slug_prefix_rules`. 🆕 It also shows the event title now.
if labels is not None:
    residual = labels.loc[labels['sport'].eq(OTHER), ['id', 'question', 'event_title', 'slug']]
    other_audit_sample = residual.sample(n=min(1000, len(residual)), random_state=20261028)
    print(f'Residual audit: sampled {len(other_audit_sample):,} of {len(residual):,} unclassified markets; showing first 50.')
    display(other_audit_sample.head(50))


Residual audit: sampled 17 of 17 unclassified markets; showing first 50.


,id,question,event_title,slug
4,251955,Haaland to break PL goal record this season?,Haaland to break PL goal record this season?,haaland-to-break-pl-goal-record-this-season
145,253360,Devin Haney vs. Regis Prograis,Devin Haney vs. Regis Prograis,devin-haney-vs-regis-prograis
3,251952,Musk vs. Zuck: Will the richer man win?,Musk vs. Zuck: Will the richer man win?,musk-vs-zuck-will-the-richer-man-win
142,253203,Will Josh Giddey's contract be terminated?,Will Josh Giddey's contract be terminated?,will-josh-giddeys-contract-be-terminated
221,253951,Will someone else be the next head coach of th...,Next Michigan head coach?,will-someone-else-be-the-next-head-coach-of-th...
2,251951,Musk vs. Zuck,Musk vs. Zuck,musk-vs-zuck-apr-1
291,254525,Ansem vs Bitlord,Ansem vs Bitlord,ansem-vs-bitlord
217,253943,Will Sherrone Moore be the next head coach of ...,Next Michigan head coach?,will-sherrone-moore-be-the-next-head-coach-of-...
104,253102,Will Draymond Green be suspended for 10+ games?,Will Draymond Green be suspended for 10+ games?,will-draymond-green-be-suspended-for-10-games
219,253945,Will Chris Klieman be the next head coach of t...,Next Michigan head coach?,will-chris-klieman-be-the-next-head-coach-of-t...


In [24]:
# 🆕 Final deliverable: one row per usable settled binary market. First 12 columns = original forecast file.
if not FORECASTS_CSV.exists():
    print('Forecast CSV not built yet.')
else:
    forecasts = pd.read_csv(FORECASTS_CSV, dtype={'market_id': 'string'})
    print(f'{len(forecasts):,} forecasts, {forecasts["market_id"].nunique():,} unique markets')
    print('Columns:', list(forecasts.columns))
    if 'sport' in forecasts.columns:
        print(f'events: {forecasts["event_id"].nunique():,}')
        display(forecasts.head())
        display(forecasts['sport'].value_counts().rename('forecasts').to_frame())
    else:
        print('These forecasts were built from a master CSV without event / sport columns: rebuild the master CSV '
              '(FORCE_EXPORT = True) and the forecasts are rebuilt automatically.')


57 forecasts, 57 unique markets
Columns: ['market_id', 'question', 'slug', 'first_outcome', 'probability', 'actual', 'hours_before_close', 'latest_hour_trade_records', 'latest_hour_trade_side_notional_usdc', 'volumeNum', 'liquidityNum', 'spread', 'event_id', 'event_title', 'event_slug', 'event_ticker', 'event_sport', 'sport', 'sport_source', 'sport_market_only']
events: 29


,market_id,question,slug,first_outcome,probability,actual,hours_before_close,latest_hour_trade_records,latest_hour_trade_side_notional_usdc,volumeNum,liquidityNum,spread,event_id,event_title,event_slug,event_ticker,event_sport,sport,sport_source,sport_market_only
0,251955,Haaland to break PL goal record this season?,haaland-to-break-pl-goal-record-this-season,Yes,0.002,0,4,2,599.999996,5449.27,NaN,1.0,902321,Haaland to break PL goal record this season?,haaland-to-break-pl-goal-record-this-season,haaland-to-break-pl-goal-record-this-season,Other / unclassified,Other / unclassified,unclassified,Other / unclassified
1,251957,Will Man City win the Premier League?,will-man-city-win-the-premier-league,Yes,0.940,1,6,4,4.255300,59808.94,NaN,1.0,902323,Will Man City win the Premier League?,will-man-city-win-the-premier-league,will-man-city-win-the-premier-league,Soccer,Soccer,event_text,Soccer
2,252221,Will Aaron Rodgers play again by December 31?,will-aaron-rodgers-play-again-by-december-31-23,Yes,0.001,0,4,2,200.000000,4273.80,0.00,1.0,902469,Will Aaron Rodgers play again by...?,will-aaron-rodgers-play-again-by,will-aaron-rodgers-play-again-by,American football,American football,event_text,American football
3,252321,Will Manchester United make it out of their gr...,will-manchester-united-make-it-out-of-their-group,Yes,0.040,0,24,2,1.000000,1472.67,0.00,1.0,902531,Will Manchester United make it out of their gr...,will-manchester-united-make-it-out-of-their-group,will-manchester-united-make-it-out-of-their-group,Soccer,Soccer,event_text,Soccer
4,252698,Will a Korean team win LOL Worlds 2023?,will-a-korean-team-win-lol-worlds-2023,Yes,0.820,1,7,2,5.268288,6403.62,0.01,1.0,902700,Who will win LoL Worlds 2023?,who-will-win-lol-worlds-2023,who-will-win-lol-worlds-2023,Esports,Esports,event_text,Esports


,forecasts
sport,
American football,38
Combat sports,5
Soccer,4
Basketball,4
Esports,2
Other / unclassified,1
Chess,1
Cricket,1
Tennis,1
